[Mohit Saharan](https://linkedin.com/in/msaharan), 20260521, draft notebook

Apache 2.0 License (see github.com/msaharan/dsaiengineering/LICENSE)

# Cross-Sectional Equity Return Ranking with GPU XGBoost, TabPFN, and TabICL

This notebook builds a public-data research workflow for cross-sectional US equity return ranking. The task is to score stocks at a monthly signal date by their relative forward return inside an eligible liquid large-cap universe. The main label is top-quintile next-month membership, and the ranking diagnostics also preserve continuous forward returns and within-month ranks.

The workflow is designed as an inspectable research notebook rather than a trading system. It uses a static current large-cap universe from public data, which introduces survivorship and index-membership limitations. Public fundamentals are fetched and analyzed only as a separate caveated diagnostic path; the headline comparison uses price, volume, liquidity, and market-regime features. The notebook is educational research, not investment advice, and does not claim that any score is suitable for live trading without separate production-grade data, execution, risk, and compliance review.

The default model path is GPU-first: deterministic finance baselines, a GPU XGBoost ranker, a GPU XGBoost top-quintile classifier with broad randomized tuning, a guarded direct TabPFN scorer, and a memory-aware guarded direct TabICL scorer when authentication, checkpoints, and GPU resources are available. CPU-only baselines are disabled by default because they can make Kaggle iteration slow without improving the main GPU workflow.

All important outputs are written to `cross_sectional_equity_return_ranking_20260521_outputs/` and zipped in the final section. Use those files for analysis because notebook editor output can be truncated.

## How to Run This Notebook on Kaggle

- Use `FAST_MODE = True` for a short smoke test of the full workflow.
- Use `FAST_MODE = False` for the production research run on a GPU notebook.
- The default run expects GPU XGBoost. If no CUDA device is detected, the notebook records that condition and disables GPU-only optional paths.
- Direct TabPFN and TabICL are guarded by authentication/checkpoint and runtime checks. If either path is unavailable, the notebook saves the reason and continues with the completed model paths.
- Public yfinance fundamentals are diagnostic only. They are not used as headline evidence because they are not a substitute for an institutional point-in-time fundamentals database.

## 0. Setup, Configuration, and Artifact Helpers

This section installs runtime dependencies, defines the research contract, creates the artifact directory, captures environment information, and sets the run mode. The configuration is intentionally explicit so a Kaggle run can be audited from saved CSV and Markdown files.

In [ ]:
# Kaggle / Colab setup.
# Run this cell once. If imports still fail after installation, restart the notebook session and continue below.

import subprocess
import sys

PYPI_PACKAGES = [
    "xgboost>=2.0.0",
    "rich",
    "yfinance>=0.2.40",
    "cupy-cuda12x",
    "tqdm",
    "scipy",
    "tabpfn==7.1.1",
    "tabicl",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PYPI_PACKAGES], check=True)

import gc
import importlib.metadata as importlib_metadata
import json
import math
import os
from pathlib import Path
import platform
import shutil
import time
import warnings
from traceback import format_exception_only
from urllib.parse import urlencode

import cupy as cp
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import yfinance as yf

from IPython.display import display
from rich.console import Console
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import ParameterSampler
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from xgboost import XGBClassifier, XGBRanker

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="The `cv='prefit'` option is deprecated", category=FutureWarning)

console = Console()
SEED = 42
np.random.seed(SEED)

FAST_MODE = False
DATA_START_DATE = "2010-01-01"
DATA_END_DATE = "2026-05-21"  # yfinance end date is exclusive.
FRED_DOWNLOAD_START_DATE = "1990-01-01"

FULL_UNIVERSE = [
    "AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "GOOG", "AVGO", "TSLA", "BRK-B",
    "JPM", "LLY", "V", "MA", "NFLX", "XOM", "COST", "WMT", "PG", "JNJ",
    "HD", "ABBV", "BAC", "KO", "PM", "CRM", "ORCL", "CVX", "WFC", "CSCO",
    "ABT", "MCD", "IBM", "GE", "LIN", "MRK", "T", "PEP", "ACN", "VZ",
    "ISRG", "NOW", "QCOM", "INTU", "AMD", "TXN", "UBER", "CAT", "DIS", "AMGN",
    "PFE", "NEE", "UNP", "GS", "SPGI", "RTX", "LOW", "AXP", "BKNG", "PGR",
    "HON", "TMO", "BLK", "SYK", "ETN", "TJX", "AMAT", "SCHW", "C", "BSX",
    "GILD", "DE", "PANW", "ADBE", "COP", "DHR", "LMT", "MDT", "VRTX", "ADI",
    "CB", "MMC", "PLD", "MU", "UPS", "BMY", "SBUX", "LRCX", "KLAC", "SO",
    "NKE", "BA", "MO", "ELV", "ICE", "CI", "SHW", "DUK", "MCO", "ZTS",
]

FAST_UNIVERSE = [
    "AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "JPM", "LLY", "V", "XOM",
    "COST", "WMT", "PG", "JNJ", "HD", "BAC", "KO", "CRM", "ORCL", "CVX",
]

SYMBOLS = FAST_UNIVERSE if FAST_MODE else FULL_UNIVERSE
BENCHMARK_SYMBOL = "SPY"
MARKET_REGIME_SYMBOLS = ["SPY", "QQQ", "IWM", "TLT", "HYG", "GLD"]
DOWNLOAD_SYMBOLS = sorted(set(SYMBOLS + [BENCHMARK_SYMBOL] + MARKET_REGIME_SYMBOLS))

SIGNAL_FREQUENCY = "monthly"
TARGET_HORIZON_MONTHS = 1
TOP_QUANTILE = 0.20
LONG_SHORT_BOTTOM_QUANTILE = 0.20
MIN_MONTHLY_ELIGIBLE_ASSETS = 10 if FAST_MODE else 40
MIN_SYMBOL_OBSERVATIONS = 756 if not FAST_MODE else 252
MIN_NONMISSING_FEATURE_RATE = 0.65
FEATURE_MAX_MISSING_RATE = 0.35
REPORTING_LAG_DAYS = 75
RUN_FUNDAMENTAL_DIAGNOSTIC = True
RUN_CPU_LOGISTIC_BASELINE = False
RUN_DIRECT_TABPFN = True
ALLOW_TABPFN_SKIP_IF_UNAVAILABLE = True
RUN_DIRECT_TABICL = True
ALLOW_TABICL_SKIP_IF_UNAVAILABLE = True
RUN_XGBOOST_RANKER = True
RUN_XGBOOST_CLASSIFIER = True
RUN_FUNDAMENTAL_XGBOOST_DIAGNOSTIC = True
SAVE_FULL_PREDICTION_SCORES = True

MODEL_SELECTION_END_DATE = "2018-12-31"
CALIBRATION_END_DATE = "2020-12-31"
HOLDOUT_START_DATE = "2021-01-01"
WALK_FORWARD_CV_SPLITS = 6
MIN_CV_TRAIN_POSITIVES = 50 if not FAST_MODE else 8
MIN_CV_VALIDATION_POSITIVES = 10 if not FAST_MODE else 2
EMBARGO_MONTHS = 1

XGBOOST_DEVICE = "cuda"
XGBOOST_TREE_METHOD = "hist"
XGBOOST_CLASSIFIER_TUNING_ITERATIONS = 160 if not FAST_MODE else 10
XGBOOST_RANKER_TUNING_ITERATIONS = 120 if not FAST_MODE else 8
XGBOOST_FINAL_N_ESTIMATORS = 1600 if not FAST_MODE else 250
XGBOOST_SEARCH_EARLY_STOPPING_ROUNDS = 75 if not FAST_MODE else 20
FUNDAMENTAL_DIAGNOSTIC_N_ESTIMATORS = 600 if not FAST_MODE else 120
N_TFM_ESTIMATORS = 8 if not FAST_MODE else 2
N_TABICL_ESTIMATORS = 4 if not FAST_MODE else 1
TABPFN_CONTEXT_MAX_ROWS = 12000 if not FAST_MODE else 1500
TABICL_CONTEXT_MAX_ROWS = 3000 if not FAST_MODE else 800
TABICL_CHECKPOINT_VERSION = "tabicl-classifier-v2-20260212.ckpt"
PREDICTION_CHUNK_SIZE = 8192
BOOTSTRAP_ITERATIONS = 400 if not FAST_MODE else 50
TRANSACTION_COST_BPS = 10.0
TRANSACTION_COST_SENSITIVITY_BPS = [0.0, 5.0, 10.0, 25.0, 50.0]
MARKET_IMPACT_NOTIONAL_USD = 1_000_000.0
ALLOW_CPU_XGBOOST_FALLBACK = False
TOP_K_PORTFOLIO = None  # If None, use top quintile count per month.

ARTIFACT_DIR = Path("cross_sectional_equity_return_ranking_20260521_outputs")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
artifact_manifest_rows = []
model_rows = []
model_errors = []
prediction_frames = []
figures = []


def short_error(exc):
    return "".join(format_exception_only(type(exc), exc)).strip().replace("\n", " ")[:600]


def installed_version(*package_names):
    for package_name in package_names:
        try:
            return importlib_metadata.version(package_name)
        except importlib_metadata.PackageNotFoundError:
            continue
    return "not installed"


def safe_string(value):
    if value is None:
        return ""
    if isinstance(value, (list, tuple, set)):
        return ", ".join(str(item) for item in value)
    return str(value)


def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def artifact_json(value):
    return json.dumps(json_ready(value), sort_keys=True)


def register_artifact(path, kind, description=""):
    path = Path(path)
    artifact_manifest_rows.append(
        {
            "path": str(path),
            "filename": path.name,
            "kind": kind,
            "description": description,
            "exists": path.exists(),
            "size_bytes": path.stat().st_size if path.exists() else np.nan,
        }
    )
    return path


def save_artifact_table(table, filename, index=False, description=""):
    output_path = ARTIFACT_DIR / filename
    table.to_csv(output_path, index=index)
    register_artifact(output_path, "table", description)
    print(f"Saved {output_path}")
    return output_path


def save_text_artifact(filename, text, description=""):
    output_path = ARTIFACT_DIR / filename
    output_path.write_text(str(text), encoding="utf-8")
    register_artifact(output_path, "text", description)
    print(f"Saved {output_path}")
    return output_path


def save_figure_artifact(fig, filename, description="", dpi=160):
    output_path = ARTIFACT_DIR / filename
    fig.savefig(output_path, dpi=dpi, bbox_inches="tight")
    register_artifact(output_path, "figure", description)
    figures.append(str(output_path))
    print(f"Saved {output_path}")
    return output_path


def y_to_numpy(y):
    if hasattr(y, "to_numpy"):
        return y.to_numpy(dtype=np.int32)
    return np.asarray(y, dtype=np.int32)


def to_numpy_float32(X):
    if hasattr(X, "to_numpy"):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)


def to_cupy_float32(X):
    return cp.asarray(to_numpy_float32(X))


def take_rows(X, row_indices):
    if hasattr(X, "iloc"):
        return X.iloc[row_indices]
    return X[row_indices]


def cleanup_runtime_memory(stage="cleanup"):
    gc.collect()
    try:
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    except Exception:
        pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except RuntimeError:
            pass


CUDA_DEVICE_COUNT = torch.cuda.device_count()
if CUDA_DEVICE_COUNT < 1:
    print("CUDA was not detected. GPU-only model paths are disabled for this run.")
    XGBOOST_DEVICE = "cpu"
    RUN_DIRECT_TABPFN = False
    RUN_DIRECT_TABICL = False
    RUN_XGBOOST_RANKER = False
    RUN_XGBOOST_CLASSIFIER = bool(ALLOW_CPU_XGBOOST_FALLBACK)

TABPFN_DEVICE = [f"cuda:{idx}" for idx in range(CUDA_DEVICE_COUNT)] if CUDA_DEVICE_COUNT else "cpu"
TABICL_DEVICE = "cuda:0" if CUDA_DEVICE_COUNT else "cpu"

configuration_summary = pd.DataFrame(
    [
        {"Parameter": "FAST_MODE", "Value": FAST_MODE},
        {"Parameter": "DATA_START_DATE", "Value": DATA_START_DATE},
        {"Parameter": "DATA_END_DATE", "Value": DATA_END_DATE},
        {"Parameter": "SYMBOL_COUNT", "Value": len(SYMBOLS)},
        {"Parameter": "SYMBOLS", "Value": safe_string(SYMBOLS)},
        {"Parameter": "BENCHMARK_SYMBOL", "Value": BENCHMARK_SYMBOL},
        {"Parameter": "TARGET_HORIZON_MONTHS", "Value": TARGET_HORIZON_MONTHS},
        {"Parameter": "TOP_QUANTILE", "Value": TOP_QUANTILE},
        {"Parameter": "MIN_MONTHLY_ELIGIBLE_ASSETS", "Value": MIN_MONTHLY_ELIGIBLE_ASSETS},
        {"Parameter": "MODEL_SELECTION_END_DATE", "Value": MODEL_SELECTION_END_DATE},
        {"Parameter": "CALIBRATION_END_DATE", "Value": CALIBRATION_END_DATE},
        {"Parameter": "HOLDOUT_START_DATE", "Value": HOLDOUT_START_DATE},
        {"Parameter": "WALK_FORWARD_CV_SPLITS", "Value": WALK_FORWARD_CV_SPLITS},
        {"Parameter": "EMBARGO_MONTHS", "Value": EMBARGO_MONTHS},
        {"Parameter": "RUN_DIRECT_TABPFN", "Value": RUN_DIRECT_TABPFN},
        {"Parameter": "RUN_DIRECT_TABICL", "Value": RUN_DIRECT_TABICL},
        {"Parameter": "ALLOW_TABICL_SKIP_IF_UNAVAILABLE", "Value": ALLOW_TABICL_SKIP_IF_UNAVAILABLE},
        {"Parameter": "RUN_XGBOOST_RANKER", "Value": RUN_XGBOOST_RANKER},
        {"Parameter": "RUN_XGBOOST_CLASSIFIER", "Value": RUN_XGBOOST_CLASSIFIER},
        {"Parameter": "RUN_FUNDAMENTAL_DIAGNOSTIC", "Value": RUN_FUNDAMENTAL_DIAGNOSTIC},
        {"Parameter": "RUN_CPU_LOGISTIC_BASELINE", "Value": RUN_CPU_LOGISTIC_BASELINE},
        {"Parameter": "XGBOOST_DEVICE", "Value": XGBOOST_DEVICE},
        {"Parameter": "TABPFN_DEVICE", "Value": safe_string(TABPFN_DEVICE)},
        {"Parameter": "TABICL_DEVICE", "Value": TABICL_DEVICE},
        {"Parameter": "XGBOOST_CLASSIFIER_TUNING_ITERATIONS", "Value": XGBOOST_CLASSIFIER_TUNING_ITERATIONS},
        {"Parameter": "XGBOOST_RANKER_TUNING_ITERATIONS", "Value": XGBOOST_RANKER_TUNING_ITERATIONS},
        {"Parameter": "N_TFM_ESTIMATORS", "Value": N_TFM_ESTIMATORS},
        {"Parameter": "N_TABICL_ESTIMATORS", "Value": N_TABICL_ESTIMATORS},
        {"Parameter": "TABICL_CONTEXT_MAX_ROWS", "Value": TABICL_CONTEXT_MAX_ROWS},
        {"Parameter": "TABICL_CHECKPOINT_VERSION", "Value": TABICL_CHECKPOINT_VERSION},
        {"Parameter": "TABPFN_CONTEXT_MAX_ROWS", "Value": TABPFN_CONTEXT_MAX_ROWS},
        {"Parameter": "TRANSACTION_COST_BPS", "Value": TRANSACTION_COST_BPS},
        {"Parameter": "TRANSACTION_COST_SENSITIVITY_BPS", "Value": safe_string(TRANSACTION_COST_SENSITIVITY_BPS)},
        {"Parameter": "MARKET_IMPACT_NOTIONAL_USD", "Value": MARKET_IMPACT_NOTIONAL_USD},
        {"Parameter": "ALLOW_CPU_XGBOOST_FALLBACK", "Value": ALLOW_CPU_XGBOOST_FALLBACK},
    ]
)

environment_summary = pd.DataFrame(
    [
        {"Component": "Python", "Version": platform.python_version()},
        {"Component": "CUDA devices", "Version": CUDA_DEVICE_COUNT},
        {"Component": "pandas", "Version": installed_version("pandas")},
        {"Component": "numpy", "Version": installed_version("numpy")},
        {"Component": "scikit-learn", "Version": sklearn.__version__},
        {"Component": "xgboost", "Version": installed_version("xgboost")},
        {"Component": "torch", "Version": torch.__version__},
        {"Component": "cupy", "Version": installed_version("cupy-cuda12x", "cupy")},
        {"Component": "yfinance", "Version": installed_version("yfinance")},
        {"Component": "tabpfn", "Version": installed_version("tabpfn")},
        {"Component": "tabicl", "Version": installed_version("tabicl")},
    ]
)

display(configuration_summary)
display(environment_summary)
save_artifact_table(configuration_summary, "configuration_summary.csv", description="Notebook configuration for this run.")
save_artifact_table(environment_summary, "environment_summary.csv", description="Runtime package and device summary.")

method_contract = f"""
# Cross-Sectional Equity Return Ranking Workflow Contract

Universe: static liquid US large-cap research list with {len(SYMBOLS)} configured symbols in this run. This is a survivorship-biased public universe and is not equivalent to a point-in-time index membership database.
Task: score each eligible stock at a monthly signal date for relative next-month return ranking inside the same month and eligible universe.
Headline target: top-quintile next-month forward return membership. Continuous forward return and within-month ranks are retained for ranking diagnostics.
Feature policy: headline features use price, volume, liquidity, high-low range, rolling beta, cross-sectional ranks, and market-regime features available at or before the signal date. Ticker identity is excluded from headline features. Sector metadata is retained for exposure diagnostics and optional sector dummies, not as a claim of point-in-time sector classification quality.
Fundamental policy: public yfinance fundamentals are diagnostic only. They are not used in headline evidence because they are not a substitute for institutional point-in-time fundamentals with explicit filing availability timestamps.
Validation: chronological monthly splits, walk-forward cross-validation for model selection, and an embargo around the forward-return horizon. No random train/test splits are used.
Models: deterministic finance baselines, equal-weight and random references, GPU XGBoost ranker, GPU XGBoost classifier, guarded direct TabPFN scorer, optional disabled CPU baselines, and a memory-aware guarded TabICL run with sampled context to manage checkpoint and GPU memory risk.
Portfolio translation: score-driven long top basket and long-short top-minus-bottom diagnostics with turnover, transaction-cost sensitivity, active-return comparison against equal-weight, sector concentration, sector-neutral portfolio diagnostics, and simple liquidity participation proxies. These are research diagnostics rather than execution-ready strategy claims.
Interpretation: educational research workflow; not investment advice; not a deployable trading recommendation.
""".strip()
save_text_artifact("method_contract.md", method_contract, description="Plain-language data, modeling, validation, and interpretation contract.")

print(f"Run mode: {'FAST smoke test' if FAST_MODE else 'full research run'}")
print(f"Configured universe size: {len(SYMBOLS)}")
print(f"CUDA devices: {CUDA_DEVICE_COUNT}; XGBoost device: {XGBOOST_DEVICE}; TabPFN device: {TABPFN_DEVICE}; TabICL device: {TABICL_DEVICE}")

## 1. Load Public Equity, Market-Regime, and Fundamentals Data

The main dataset comes from public OHLCV data downloaded through yfinance. The notebook records symbol coverage, dropped symbols, market-regime source availability, and fundamental diagnostic availability. Public data improves reproducibility, but the limitations are part of the research contract.

In [ ]:
def normalize_yfinance_download(raw, symbols):
    if raw.empty:
        raise RuntimeError("yfinance returned an empty price frame.")
    if isinstance(raw.columns, pd.MultiIndex):
        if raw.columns.names[0] == "Price":
            stacked = raw.stack(level=1, future_stack=True).rename_axis(["date", "symbol"]).reset_index()
        else:
            stacked = raw.stack(level=0, future_stack=True).rename_axis(["date", "symbol"]).reset_index()
    else:
        if len(symbols) != 1:
            raise RuntimeError("Expected a MultiIndex yfinance result for multiple symbols.")
        stacked = raw.copy()
        stacked["symbol"] = symbols[0]
        stacked = stacked.reset_index().rename(columns={"Date": "date"})
    stacked.columns = [str(column).lower().replace(" ", "_") for column in stacked.columns]
    rename_map = {"adj_close": "adj_close", "close": "close", "open": "open", "high": "high", "low": "low", "volume": "volume"}
    stacked = stacked.rename(columns=rename_map)
    stacked["date"] = pd.to_datetime(stacked["date"]).dt.tz_localize(None)
    stacked["symbol"] = stacked["symbol"].astype(str)
    required = ["date", "symbol", "open", "high", "low", "close", "adj_close", "volume"]
    missing = [column for column in required if column not in stacked.columns]
    if missing:
        raise RuntimeError(f"Normalized yfinance frame is missing required columns: {missing}")
    return stacked[required].sort_values(["symbol", "date"]).reset_index(drop=True)


def download_prices(symbols, start, end):
    frames = []
    chunk_size = 25 if not FAST_MODE else 10
    chunks = [symbols[idx : idx + chunk_size] for idx in range(0, len(symbols), chunk_size)]
    for chunk in tqdm(chunks, desc="Downloading OHLCV chunks", unit="chunk"):
        raw = yf.download(chunk, start=start, end=end, auto_adjust=False, actions=False, progress=False, threads=True)
        frames.append(normalize_yfinance_download(raw, chunk))
    return pd.concat(frames, ignore_index=True).drop_duplicates(["date", "symbol"]).sort_values(["symbol", "date"])


price_data = download_prices(DOWNLOAD_SYMBOLS, DATA_START_DATE, DATA_END_DATE)
price_data["adj_factor"] = price_data["adj_close"] / price_data["close"].replace(0, np.nan)
for column in ["open", "high", "low", "close"]:
    price_data[f"adj_{column}"] = price_data[column] * price_data["adj_factor"]

coverage_summary = (
    price_data.groupby("symbol")
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        observations=("adj_close", "count"),
        missing_adj_close=("adj_close", lambda values: int(values.isna().sum())),
        median_dollar_volume=("volume", lambda values: np.nan),
    )
    .reset_index()
)

dollar_volume = price_data.assign(dollar_volume=price_data["adj_close"] * price_data["volume"])
median_dv = dollar_volume.groupby("symbol")["dollar_volume"].median().rename("median_dollar_volume").reset_index()
coverage_summary = coverage_summary.drop(columns=["median_dollar_volume"]).merge(median_dv, on="symbol", how="left")
coverage_summary["configured_universe"] = coverage_summary["symbol"].isin(SYMBOLS)
coverage_summary["coverage_pass"] = coverage_summary["observations"] >= MIN_SYMBOL_OBSERVATIONS
coverage_summary["in_model_universe"] = coverage_summary["configured_universe"] & coverage_summary["coverage_pass"]
model_symbols = sorted(coverage_summary.loc[coverage_summary["in_model_universe"], "symbol"].tolist())
if len(model_symbols) < MIN_MONTHLY_ELIGIBLE_ASSETS:
    raise RuntimeError(f"Only {len(model_symbols)} configured symbols passed coverage; need at least {MIN_MONTHLY_ELIGIBLE_ASSETS}.")

save_artifact_table(coverage_summary, "price_coverage_summary.csv", description="OHLCV coverage and model-universe eligibility by symbol.")
display(coverage_summary.sort_values(["in_model_universe", "median_dollar_volume"], ascending=[False, False]).head(20))
print(f"Model universe after coverage filter: {len(model_symbols)} symbols")


def fred_csv_url(series_id, observation_start=FRED_DOWNLOAD_START_DATE):
    query = urlencode({"id": series_id, "observation_start": observation_start})
    return f"https://fred.stlouisfed.org/graph/fredgraph.csv?{query}"


def load_fred_series(series_map):
    frames = []
    rows = []
    for series_id, name in tqdm(series_map.items(), desc="Downloading FRED series", unit="series"):
        try:
            frame = pd.read_csv(fred_csv_url(series_id))
            frame.columns = ["date", name]
            frame["date"] = pd.to_datetime(frame["date"])
            frame[name] = pd.to_numeric(frame[name].replace(".", np.nan), errors="coerce")
            frames.append(frame)
            rows.append({"series_id": series_id, "name": name, "status": "loaded", "non_missing": int(frame[name].notna().sum()), "error": ""})
        except Exception as exc:
            rows.append({"series_id": series_id, "name": name, "status": "failed", "non_missing": 0, "error": short_error(exc)})
    if not frames:
        return pd.DataFrame(columns=["date"]), pd.DataFrame(rows)
    out = frames[0]
    for frame in frames[1:]:
        out = out.merge(frame, on="date", how="outer")
    return out.sort_values("date"), pd.DataFrame(rows)


FRED_SERIES = {
    "DGS10": "treasury_10y_yield",
    "DGS2": "treasury_2y_yield",
    "T10Y2Y": "yield_curve_10y_2y",
    "DFF": "fed_funds_rate",
    "DTB3": "t_bill_3m",
    "BAMLH0A0HYM2": "high_yield_oas",
}

fred_daily, fred_availability = load_fred_series(FRED_SERIES)
save_artifact_table(fred_availability, "fred_series_availability_summary.csv", description="FRED market-regime source availability.")

try:
    vix_daily = pd.read_csv("https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv")
    vix_daily.columns = [column.strip().lower().replace(" ", "_") for column in vix_daily.columns]
    vix_daily = vix_daily.rename(columns={"date": "date", "close": "vix_close"})
    vix_daily["date"] = pd.to_datetime(vix_daily["date"])
    vix_daily = vix_daily[["date", "vix_close"]]
    vix_status = pd.DataFrame([{"source": "Cboe VIX history", "status": "loaded", "rows": len(vix_daily), "error": ""}])
except Exception as exc:
    vix_daily = pd.DataFrame(columns=["date", "vix_close"])
    vix_status = pd.DataFrame([{"source": "Cboe VIX history", "status": "failed", "rows": 0, "error": short_error(exc)}])
save_artifact_table(vix_status, "vix_source_status.csv", description="VIX source status.")


def ticker_info_value(info, keys):
    for key in keys:
        value = info.get(key)
        if value not in (None, ""):
            return value
    return np.nan


def fetch_fundamental_diagnostics(symbols):
    summary_rows = []
    quarterly_rows = []
    if not RUN_FUNDAMENTAL_DIAGNOSTIC:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame([{"status": "skipped", "symbol": "ALL", "error": "RUN_FUNDAMENTAL_DIAGNOSTIC=False"}])
    for symbol in tqdm(symbols, desc="Fetching yfinance fundamentals", unit="symbol"):
        try:
            ticker = yf.Ticker(symbol)
            info = ticker.get_info()
            summary_rows.append(
                {
                    "symbol": symbol,
                    "sector": ticker_info_value(info, ["sector"]),
                    "industry": ticker_info_value(info, ["industry"]),
                    "market_cap_current": ticker_info_value(info, ["marketCap"]),
                    "trailing_pe_current": ticker_info_value(info, ["trailingPE"]),
                    "forward_pe_current": ticker_info_value(info, ["forwardPE"]),
                    "price_to_book_current": ticker_info_value(info, ["priceToBook"]),
                    "profit_margins_current": ticker_info_value(info, ["profitMargins"]),
                    "return_on_equity_current": ticker_info_value(info, ["returnOnEquity"]),
                    "debt_to_equity_current": ticker_info_value(info, ["debtToEquity"]),
                    "dividend_yield_current": ticker_info_value(info, ["dividendYield"]),
                }
            )
            quarterly_financials = ticker.quarterly_financials
            quarterly_balance = ticker.quarterly_balance_sheet
            quarterly_cashflow = ticker.quarterly_cashflow
            selected = {}
            source_frames = {
                "financials": quarterly_financials,
                "balance_sheet": quarterly_balance,
                "cashflow": quarterly_cashflow,
            }
            wanted_rows = {
                "Total Revenue": "total_revenue",
                "Net Income": "net_income",
                "EBITDA": "ebitda",
                "Gross Profit": "gross_profit",
                "Total Assets": "total_assets",
                "Total Debt": "total_debt",
                "Stockholders Equity": "stockholders_equity",
                "Operating Cash Flow": "operating_cash_flow",
                "Free Cash Flow": "free_cash_flow",
            }
            for frame in source_frames.values():
                if frame is None or frame.empty:
                    continue
                for raw_name, clean_name in wanted_rows.items():
                    if raw_name in frame.index:
                        selected[clean_name] = frame.loc[raw_name]
            if selected:
                q = pd.DataFrame(selected)
                q.index = pd.to_datetime(q.index)
                q = q.reset_index().rename(columns={"index": "fiscal_period_end"})
                q["symbol"] = symbol
                quarterly_rows.append(q)
        except Exception as exc:
            summary_rows.append({"symbol": symbol, "sector": np.nan, "industry": np.nan, "fundamental_error": short_error(exc)})
    summary = pd.DataFrame(summary_rows)
    quarterly = pd.concat(quarterly_rows, ignore_index=True) if quarterly_rows else pd.DataFrame()
    status = pd.DataFrame(
        [
            {"status": "complete", "symbols_requested": len(symbols), "summary_rows": len(summary), "quarterly_rows": len(quarterly), "error": ""}
        ]
    )
    return summary, quarterly, status


fundamental_summary, quarterly_fundamentals, fundamental_status = fetch_fundamental_diagnostics(model_symbols)
save_artifact_table(fundamental_status, "fundamental_fetch_status.csv", description="Public fundamentals diagnostic fetch status.")
if len(fundamental_summary) > 0:
    save_artifact_table(fundamental_summary, "fundamental_current_summary.csv", description="Current yfinance fundamental summary; diagnostic only, not headline evidence.")
if len(quarterly_fundamentals) > 0:
    save_artifact_table(quarterly_fundamentals, "fundamental_quarterly_raw.csv", description="Quarterly yfinance fundamentals; diagnostic only.")

data_source_summary = pd.DataFrame(
    [
        {"source": "yfinance OHLCV", "rows": len(price_data), "symbols": price_data["symbol"].nunique(), "status": "loaded"},
        {"source": "FRED", "rows": len(fred_daily), "symbols": fred_daily.shape[1] - 1 if len(fred_daily) else 0, "status": "loaded" if len(fred_daily) else "empty"},
        {"source": "Cboe VIX", "rows": len(vix_daily), "symbols": 1 if len(vix_daily) else 0, "status": vix_status.iloc[0]["status"]},
        {"source": "yfinance fundamentals", "rows": len(fundamental_summary), "symbols": fundamental_summary["symbol"].nunique() if len(fundamental_summary) else 0, "status": fundamental_status.iloc[0]["status"]},
    ]
)
save_artifact_table(data_source_summary, "data_source_summary.csv", description="Loaded public data source summary.")
display(data_source_summary)

## 2. Build Monthly Features and Cross-Sectional Targets

The feature table is an asset-month panel. Features are computed from data available at or before the monthly signal date. Forward returns are computed separately and excluded from model features. The target is defined within each month, so the model is evaluated as a cross-sectional ranker rather than a single-stock price forecaster.

In [ ]:
def wide_panel(frame, value):
    panel = frame.pivot(index="date", columns="symbol", values=value).sort_index()
    return panel


model_price_data = price_data[price_data["symbol"].isin(sorted(set(model_symbols + [BENCHMARK_SYMBOL] + MARKET_REGIME_SYMBOLS)))].copy()
adj_close = wide_panel(model_price_data, "adj_close")
adj_open = wide_panel(model_price_data, "adj_open")
adj_high = wide_panel(model_price_data, "adj_high")
adj_low = wide_panel(model_price_data, "adj_low")
volume = wide_panel(model_price_data, "volume")
returns = adj_close.pct_change(fill_method=None)
log_returns = np.log(adj_close).diff()
benchmark_returns = returns[BENCHMARK_SYMBOL].copy() if BENCHMARK_SYMBOL in returns.columns else returns[model_symbols].mean(axis=1)

monthly_signal_dates = adj_close[model_symbols].resample("ME").last().index
monthly_signal_dates = monthly_signal_dates[monthly_signal_dates >= pd.Timestamp(DATA_START_DATE)]
monthly_first_open = adj_open.resample("MS").first()
monthly_close = adj_close.resample("ME").last()

fred_monthly = pd.DataFrame(index=monthly_signal_dates)
if len(fred_daily) > 0:
    fred_monthly = fred_daily.set_index("date").sort_index().reindex(monthly_signal_dates, method="ffill")
    fred_monthly.index = monthly_signal_dates
if len(vix_daily) > 0:
    vix_monthly = vix_daily.set_index("date").sort_index().reindex(monthly_signal_dates, method="ffill")
    vix_monthly.index = monthly_signal_dates
    fred_monthly = fred_monthly.join(vix_monthly, how="left")

def month_end_last(series):
    return series.resample("ME").last().reindex(monthly_signal_dates)


market_feature_frame = pd.DataFrame(index=monthly_signal_dates)
for symbol in [item for item in MARKET_REGIME_SYMBOLS if item in returns.columns]:
    market_feature_frame[f"market_{symbol.lower()}_return_21d"] = month_end_last(adj_close[symbol].pct_change(21))
    market_feature_frame[f"market_{symbol.lower()}_return_63d"] = month_end_last(adj_close[symbol].pct_change(63))
    market_feature_frame[f"market_{symbol.lower()}_rv_21d"] = month_end_last(log_returns[symbol].rolling(21).std().mul(math.sqrt(252)))
if len(fred_monthly) > 0:
    for column in fred_monthly.columns:
        market_feature_frame[f"macro_{column}"] = fred_monthly[column]
        market_feature_frame[f"macro_{column}_change_3m"] = fred_monthly[column].diff(3)


def rolling_downside_vol(series, window):
    downside = series.where(series < 0.0, 0.0)
    return downside.rolling(window).std() * math.sqrt(252)


def rolling_beta(asset_return, benchmark_return, window):
    covariance = asset_return.rolling(window).cov(benchmark_return)
    variance = benchmark_return.rolling(window).var()
    return covariance / variance.replace(0, np.nan)


def build_symbol_monthly_features(symbol):
    close = adj_close[symbol]
    high = adj_high[symbol]
    low = adj_low[symbol]
    vol = volume[symbol]
    asset_returns = returns[symbol]
    asset_log_returns = log_returns[symbol]
    frame = pd.DataFrame(index=monthly_signal_dates)
    frame["date"] = monthly_signal_dates
    frame["month"] = frame["date"].dt.to_period("M").astype(str)
    frame["symbol"] = symbol
    frame["adj_close"] = month_end_last(close)
    frame["volume"] = month_end_last(vol)
    frame["dollar_volume"] = month_end_last(close * vol)
    for window in [1, 5, 21, 63, 126, 252]:
        frame[f"return_{window}d"] = month_end_last(close.pct_change(window))
    for window in [5, 21, 63]:
        frame[f"reversal_{window}d"] = month_end_last(-close.pct_change(window))
    for window in [21, 63, 126]:
        frame[f"rv_{window}d"] = month_end_last(asset_log_returns.rolling(window).std().mul(math.sqrt(252)))
        frame[f"downside_rv_{window}d"] = month_end_last(rolling_downside_vol(asset_log_returns, window))
    for window in [63, 126, 252]:
        frame[f"drawdown_{window}d"] = month_end_last(close / close.rolling(window).max() - 1.0)
    for window in [20, 50, 100, 200]:
        frame[f"ma_distance_{window}d"] = month_end_last(close / close.rolling(window).mean() - 1.0)
    for window in [63, 126, 252]:
        frame[f"beta_spy_{window}d"] = month_end_last(rolling_beta(asset_returns, benchmark_returns, window))
    for window in [21, 63, 126]:
        frame[f"avg_volume_{window}d"] = month_end_last(vol.rolling(window).mean())
        frame[f"avg_dollar_volume_{window}d"] = month_end_last((close * vol).rolling(window).mean())
        frame[f"high_low_range_{window}d"] = month_end_last(((high - low) / close).rolling(window).mean())
    frame["illiquidity_proxy_21d"] = month_end_last((asset_returns.abs() / (close * vol).replace(0, np.nan)).rolling(21).mean())
    frame["market_beta_residual_63d"] = month_end_last((asset_returns - rolling_beta(asset_returns, benchmark_returns, 63) * benchmark_returns).rolling(63).std().mul(math.sqrt(252)))
    return frame.reset_index(drop=True)


feature_frames = [build_symbol_monthly_features(symbol) for symbol in tqdm(model_symbols, desc="Building symbol features", unit="symbol")]
model_frame = pd.concat(feature_frames, ignore_index=True)
model_frame = model_frame.merge(market_feature_frame.reset_index().rename(columns={"index": "date"}), on="date", how="left")

if len(fundamental_summary) > 0:
    sector_map = fundamental_summary.set_index("symbol")["sector"].to_dict() if "sector" in fundamental_summary else {}
    industry_map = fundamental_summary.set_index("symbol")["industry"].to_dict() if "industry" in fundamental_summary else {}
else:
    sector_map = {}
    industry_map = {}
model_frame["sector"] = model_frame["symbol"].map(sector_map).fillna("Unknown")
model_frame["industry"] = model_frame["symbol"].map(industry_map).fillna("Unknown")

# Forward open-to-open return after the signal month. Adjusted open is used for return continuity.
open_by_period = monthly_first_open.copy()
open_by_period.index = open_by_period.index.to_period("M")
forward_rows = []
for symbol in tqdm(model_symbols, desc="Building forward returns", unit="symbol"):
    symbol_open = open_by_period[symbol]
    for date in monthly_signal_dates:
        period = date.to_period("M")
        entry_period = period + 1
        exit_period = period + 2
        entry_open = symbol_open.get(entry_period, np.nan)
        exit_open = symbol_open.get(exit_period, np.nan)
        forward_return = exit_open / entry_open - 1.0 if pd.notna(entry_open) and pd.notna(exit_open) and entry_open != 0 else np.nan
        forward_rows.append({"date": date, "symbol": symbol, "forward_1m_open_return": forward_return, "entry_period": str(entry_period), "exit_period": str(exit_period)})
forward_return_frame = pd.DataFrame(forward_rows)
model_frame = model_frame.merge(forward_return_frame, on=["date", "symbol"], how="left")

# Cross-sectional labels are assigned only within the same signal month.
model_frame["forward_1m_rank"] = model_frame.groupby("date")["forward_1m_open_return"].rank(method="first", ascending=False)
model_frame["forward_1m_pct_rank"] = model_frame.groupby("date")["forward_1m_open_return"].rank(pct=True, ascending=True)
model_frame["monthly_eligible_assets"] = model_frame.groupby("date")["forward_1m_open_return"].transform(lambda values: int(values.notna().sum()))
model_frame["target_top_quintile"] = (
    model_frame.groupby("date")["forward_1m_open_return"].rank(pct=True, ascending=False) <= TOP_QUANTILE
).astype("float")
model_frame.loc[model_frame["forward_1m_open_return"].isna(), "target_top_quintile"] = np.nan
model_frame["rank_relevance_quintile"] = pd.NA
for date, group_index in tqdm(model_frame.groupby("date").groups.items(), desc="Assigning rank relevance", unit="month"):
    group = model_frame.loc[group_index, "forward_1m_open_return"]
    valid = group.dropna()
    if valid.nunique() >= 5:
        quintiles = pd.qcut(valid.rank(method="first"), q=5, labels=False, duplicates="drop")
        model_frame.loc[quintiles.index, "rank_relevance_quintile"] = quintiles.astype(int)
model_frame["rank_relevance_quintile"] = pd.to_numeric(model_frame["rank_relevance_quintile"], errors="coerce")

# Add cross-sectional ranks for model-safe numeric features.
feature_candidate_columns = [
    column
    for column in model_frame.columns
    if column not in {"date", "month", "symbol", "sector", "industry", "entry_period", "exit_period", "target_top_quintile", "rank_relevance_quintile", "monthly_eligible_assets"}
    and not column.startswith("forward_")
]
for column in tqdm(feature_candidate_columns, desc="Adding cross-sectional feature ranks", unit="feature"):
    if pd.api.types.is_numeric_dtype(model_frame[column]):
        model_frame[f"cs_rank_{column}"] = model_frame.groupby("date")[column].rank(pct=True)

# Eligibility filters for the final modeling frame.
model_frame = model_frame[model_frame["symbol"].isin(model_symbols)].copy()
model_frame = model_frame[model_frame["monthly_eligible_assets"] >= MIN_MONTHLY_ELIGIBLE_ASSETS].copy()
model_frame = model_frame.dropna(subset=["target_top_quintile", "rank_relevance_quintile", "forward_1m_open_return"])
model_frame["target_top_quintile"] = model_frame["target_top_quintile"].astype(int)
model_frame["rank_relevance_quintile"] = model_frame["rank_relevance_quintile"].astype(int)
model_frame = model_frame.sort_values(["date", "symbol"]).reset_index(drop=True)

# Build diagnostic-only fundamentals with a conservative reporting lag when quarterly data is available.
RAW_FUNDAMENTAL_COLUMNS = ["total_revenue", "net_income", "ebitda", "gross_profit", "total_assets", "total_debt", "stockholders_equity", "operating_cash_flow", "free_cash_flow"]
fundamental_feature_columns = []
if len(quarterly_fundamentals) > 0:
    quarterly = quarterly_fundamentals.copy()
    quarterly["availability_date"] = pd.to_datetime(quarterly["fiscal_period_end"]) + pd.Timedelta(days=REPORTING_LAG_DAYS)
    quarterly = quarterly.sort_values(["symbol", "availability_date"])
    merged_parts = []
    for symbol, symbol_frame in tqdm(model_frame.groupby("symbol"), desc="Merging lagged fundamentals", unit="symbol"):
        q = quarterly[quarterly["symbol"] == symbol].copy()
        if q.empty:
            merged_parts.append(symbol_frame)
            continue
        left = symbol_frame.sort_values("date")
        right = q.drop(columns=["symbol"]).sort_values("availability_date")
        merged = pd.merge_asof(left, right, left_on="date", right_on="availability_date", direction="backward")
        merged_parts.append(merged)
    model_frame = pd.concat(merged_parts, ignore_index=True).sort_values(["date", "symbol"]).reset_index(drop=True)
    raw_fundamental_columns = [column for column in RAW_FUNDAMENTAL_COLUMNS if column in model_frame.columns]
    for column in raw_fundamental_columns:
        model_frame[f"fundamental_{column}_log_abs"] = np.sign(model_frame[column]) * np.log1p(model_frame[column].abs())
        fundamental_feature_columns.append(f"fundamental_{column}_log_abs")
    if {"net_income", "total_assets"}.issubset(model_frame.columns):
        model_frame["fundamental_roa_proxy"] = model_frame["net_income"] / model_frame["total_assets"].replace(0, np.nan)
        fundamental_feature_columns.append("fundamental_roa_proxy")
    if {"total_debt", "total_assets"}.issubset(model_frame.columns):
        model_frame["fundamental_debt_to_assets_proxy"] = model_frame["total_debt"] / model_frame["total_assets"].replace(0, np.nan)
        fundamental_feature_columns.append("fundamental_debt_to_assets_proxy")
    if {"net_income", "stockholders_equity"}.issubset(model_frame.columns):
        model_frame["fundamental_roe_proxy"] = model_frame["net_income"] / model_frame["stockholders_equity"].replace(0, np.nan)
        fundamental_feature_columns.append("fundamental_roe_proxy")

for column in fundamental_feature_columns:
    model_frame[f"cs_rank_{column}"] = model_frame.groupby("date")[column].rank(pct=True)
fundamental_feature_columns = fundamental_feature_columns + [f"cs_rank_{column}" for column in fundamental_feature_columns if f"cs_rank_{column}" in model_frame.columns]

monthly_sampling_policy = pd.DataFrame(
    [
        {
            "item": "daily_feature_monthly_sampling",
            "policy": "Daily features are resampled with month-end last observation, so calendar month-end weekends and holidays use the last available trading-day value.",
        },
        {
            "item": "forward_return_convention",
            "policy": "Forward returns use adjusted open from the first trading day of the next month to the first trading day of the following month.",
        },
    ]
)
save_artifact_table(monthly_sampling_policy, "monthly_sampling_policy.csv", description="Monthly signal-date sampling and forward-return conventions.")

model_frame_head = model_frame.head(100)
save_artifact_table(model_frame_head, "model_frame_head.csv", description="First rows of the monthly modeling table.")

target_summary = (
    model_frame.groupby("date")
    .agg(
        rows=("symbol", "size"),
        eligible_assets=("monthly_eligible_assets", "max"),
        positive_rate=("target_top_quintile", "mean"),
        mean_forward_return=("forward_1m_open_return", "mean"),
        median_forward_return=("forward_1m_open_return", "median"),
    )
    .reset_index()
)
save_artifact_table(target_summary, "target_distribution_by_month.csv", description="Monthly cross-sectional target and forward-return summary.")

feature_quality_rows = []
for column in model_frame.columns:
    if pd.api.types.is_numeric_dtype(model_frame[column]):
        feature_quality_rows.append(
            {
                "column": column,
                "missing_rate": float(model_frame[column].isna().mean()),
                "non_missing": int(model_frame[column].notna().sum()),
                "unique_values": int(model_frame[column].nunique(dropna=True)),
                "min": float(model_frame[column].min(skipna=True)) if model_frame[column].notna().any() else np.nan,
                "max": float(model_frame[column].max(skipna=True)) if model_frame[column].notna().any() else np.nan,
            }
        )
feature_quality_summary = pd.DataFrame(feature_quality_rows).sort_values(["missing_rate", "column"])
save_artifact_table(feature_quality_summary, "feature_quality_summary.csv", description="Numeric column missingness and value summary.")

fundamental_caveat = """
# Public Fundamental Diagnostic Caveat

The notebook fetches available yfinance fundamentals as a diagnostic path only. These data are not treated as institutional point-in-time fundamentals. A conservative reporting lag is applied where quarterly statement dates are available, but that does not prove that each value was available to the market at the modeled signal date. The headline model comparison therefore excludes fundamental features. Fundamental diagnostics should be interpreted as sensitivity analysis and data-engineering exploration, not as clean factor evidence.
""".strip()
save_text_artifact("fundamental_diagnostic_caveat.md", fundamental_caveat, description="Caveat for public fundamental diagnostic features.")

display(target_summary.tail())
print(f"Final model frame: {len(model_frame):,} rows, {model_frame['date'].nunique()} months, {model_frame['symbol'].nunique()} symbols")
print(f"Fundamental diagnostic columns available: {len(fundamental_feature_columns)}")

## 3. Chronological Splits, Feature Policy, and Leakage Checks

The data is split by time. Model selection uses only earlier months, calibration uses the next block, and holdout starts after the calibration block. Walk-forward folds use monthly groups and an embargo so the validation month does not immediately overlap the forward-return horizon.

In [ ]:
def build_feature_columns(frame, include_fundamentals=False):
    excluded_exact = {
        "date", "month", "symbol", "sector", "industry", "entry_period", "exit_period", "target_top_quintile", "rank_relevance_quintile",
        "forward_1m_open_return", "forward_1m_rank", "forward_1m_pct_rank", "monthly_eligible_assets",
        "fiscal_period_end", "availability_date",
    }
    excluded_prefixes = ("forward_",)
    candidate_columns = []
    for column in frame.columns:
        if column in excluded_exact or column in RAW_FUNDAMENTAL_COLUMNS or column.startswith(excluded_prefixes):
            continue
        if column.startswith("cs_rank_monthly_eligible_assets"):
            continue
        if column in fundamental_feature_columns and not include_fundamentals:
            continue
        if column.startswith("fundamental_") and not include_fundamentals:
            continue
        if pd.api.types.is_numeric_dtype(frame[column]):
            candidate_columns.append(column)
    candidate_columns = [column for column in candidate_columns if not column.endswith("_current")]
    missing_rates = frame[candidate_columns].isna().mean()
    usable_columns = sorted(missing_rates[missing_rates <= FEATURE_MAX_MISSING_RATE].index.tolist())
    return usable_columns


headline_feature_columns = build_feature_columns(model_frame, include_fundamentals=False)
fundamental_augmented_feature_columns = build_feature_columns(model_frame, include_fundamentals=True)
fundamental_only_feature_columns = [column for column in fundamental_augmented_feature_columns if column in fundamental_feature_columns or column.startswith("fundamental_")]

sector_dummies = pd.get_dummies(model_frame["sector"].fillna("Unknown"), prefix="sector", dtype=np.int8)
sector_feature_columns = sector_dummies.columns.tolist()
model_frame_with_sector = pd.concat([model_frame.reset_index(drop=True), sector_dummies.reset_index(drop=True)], axis=1)
headline_with_sector_feature_columns = headline_feature_columns + sector_feature_columns

# Headline matrix excludes ticker identity and fundamentals. Sector dummies are included as broad static metadata, while sector exposure is audited separately.
ACTIVE_FEATURE_COLUMNS = headline_with_sector_feature_columns

model_dates = pd.to_datetime(model_frame_with_sector["date"])
model_selection_mask = model_dates <= pd.Timestamp(MODEL_SELECTION_END_DATE)
calibration_mask = (model_dates > pd.Timestamp(MODEL_SELECTION_END_DATE)) & (model_dates <= pd.Timestamp(CALIBRATION_END_DATE))
preholdout_mask = model_dates <= pd.Timestamp(CALIBRATION_END_DATE)
holdout_mask = model_dates >= pd.Timestamp(HOLDOUT_START_DATE)

selection_df = model_frame_with_sector.loc[model_selection_mask].copy()
calibration_df = model_frame_with_sector.loc[calibration_mask].copy()
preholdout_df = model_frame_with_sector.loc[preholdout_mask].copy()
holdout_df = model_frame_with_sector.loc[holdout_mask].copy()

if selection_df.empty or calibration_df.empty or holdout_df.empty:
    raise RuntimeError("One or more chronological split windows are empty. Adjust split dates or data start/end dates.")

X_selection_raw = selection_df[ACTIVE_FEATURE_COLUMNS]
X_calibration_raw = calibration_df[ACTIVE_FEATURE_COLUMNS]
X_preholdout_raw = preholdout_df[ACTIVE_FEATURE_COLUMNS]
X_holdout_raw = holdout_df[ACTIVE_FEATURE_COLUMNS]
y_selection = selection_df["target_top_quintile"].astype(int)
y_calibration = calibration_df["target_top_quintile"].astype(int)
y_preholdout = preholdout_df["target_top_quintile"].astype(int)
y_holdout = holdout_df["target_top_quintile"].astype(int)
y_selection_rank = selection_df["rank_relevance_quintile"].astype(int)
y_preholdout_rank = preholdout_df["rank_relevance_quintile"].astype(int)
y_calibration_rank = calibration_df["rank_relevance_quintile"].astype(int)
y_holdout_rank = holdout_df["rank_relevance_quintile"].astype(int)

selection_imputer = SimpleImputer(strategy="median", add_indicator=True)
preholdout_imputer = SimpleImputer(strategy="median", add_indicator=True)

X_selection = pd.DataFrame(selection_imputer.fit_transform(X_selection_raw), index=selection_df.index)
X_calibration = pd.DataFrame(selection_imputer.transform(X_calibration_raw), index=calibration_df.index)
X_preholdout = pd.DataFrame(preholdout_imputer.fit_transform(X_preholdout_raw), index=preholdout_df.index)
X_holdout = pd.DataFrame(preholdout_imputer.transform(X_holdout_raw), index=holdout_df.index)

selection_feature_names = [f"selection_feature_{idx:04d}" for idx in range(X_selection.shape[1])]
preholdout_feature_names = [f"preholdout_feature_{idx:04d}" for idx in range(X_preholdout.shape[1])]
X_selection.columns = selection_feature_names
X_calibration.columns = selection_feature_names
X_preholdout.columns = preholdout_feature_names
X_holdout.columns = preholdout_feature_names

imputation_summary = pd.DataFrame(
    [
        {
            "matrix_pair": "selection_to_calibration",
            "fit_rows": len(X_selection_raw),
            "transform_rows": len(X_calibration_raw),
            "raw_features": len(ACTIVE_FEATURE_COLUMNS),
            "features_after_imputation": X_selection.shape[1],
            "indicator_features": X_selection.shape[1] - len(ACTIVE_FEATURE_COLUMNS),
            "fit_window": "model_selection",
            "transform_window": "calibration",
        },
        {
            "matrix_pair": "preholdout_to_holdout",
            "fit_rows": len(X_preholdout_raw),
            "transform_rows": len(X_holdout_raw),
            "raw_features": len(ACTIVE_FEATURE_COLUMNS),
            "features_after_imputation": X_preholdout.shape[1],
            "indicator_features": X_preholdout.shape[1] - len(ACTIVE_FEATURE_COLUMNS),
            "fit_window": "preholdout",
            "transform_window": "holdout",
        },
    ]
)
save_artifact_table(imputation_summary, "imputation_policy_summary.csv", description="Separate imputation fits for selection/calibration and preholdout/holdout matrices.")

split_summary = []
for name, frame, y in [
    ("model_selection", selection_df, y_selection),
    ("calibration", calibration_df, y_calibration),
    ("preholdout", preholdout_df, y_preholdout),
    ("holdout", holdout_df, y_holdout),
]:
    split_summary.append(
        {
            "split": name,
            "rows": len(frame),
            "months": frame["date"].nunique(),
            "symbols": frame["symbol"].nunique(),
            "positive_rate": float(y.mean()),
            "start": str(frame["date"].min().date()),
            "end": str(frame["date"].max().date()),
        }
    )
split_summary = pd.DataFrame(split_summary)
save_artifact_table(split_summary, "split_summary.csv", description="Chronological split summary.")
display(split_summary)

feature_policy_summary = pd.DataFrame(
    [
        {"policy": "headline", "features_before_imputation": len(ACTIVE_FEATURE_COLUMNS), "features_after_imputation": X_preholdout.shape[1], "uses_fundamentals": False, "uses_ticker_identity": False, "uses_sector_dummies": True},
        {"policy": "fundamental_augmented_diagnostic", "features_before_imputation": len(fundamental_augmented_feature_columns), "features_after_imputation": np.nan, "uses_fundamentals": True, "uses_ticker_identity": False, "uses_sector_dummies": False},
        {"policy": "fundamental_only_diagnostic", "features_before_imputation": len(fundamental_only_feature_columns), "features_after_imputation": np.nan, "uses_fundamentals": True, "uses_ticker_identity": False, "uses_sector_dummies": False},
    ]
)
save_artifact_table(feature_policy_summary, "feature_policy_summary.csv", description="Feature policy summary.")

feature_family_summary = pd.DataFrame(
    [
        {"family": "momentum", "example_features": "return_21d, return_63d, return_126d, return_252d", "windows": "1, 5, 21, 63, 126, 252 trading days"},
        {"family": "reversal", "example_features": "reversal_5d, reversal_21d, reversal_63d", "windows": "5, 21, 63 trading days"},
        {"family": "realized_volatility", "example_features": "rv_21d, rv_63d, rv_126d", "windows": "21, 63, 126 trading days"},
        {"family": "downside_volatility", "example_features": "downside_rv_21d, downside_rv_63d, downside_rv_126d", "windows": "21, 63, 126 trading days"},
        {"family": "drawdown", "example_features": "drawdown_63d, drawdown_126d, drawdown_252d", "windows": "63, 126, 252 trading days"},
        {"family": "moving_average_distance", "example_features": "ma_distance_20d, ma_distance_50d, ma_distance_100d, ma_distance_200d", "windows": "20, 50, 100, 200 trading days"},
        {"family": "beta", "example_features": "beta_spy_63d, beta_spy_126d, beta_spy_252d", "windows": "63, 126, 252 trading days"},
        {"family": "liquidity", "example_features": "avg_volume_63d, avg_dollar_volume_63d, illiquidity_proxy_21d", "windows": "21, 63, 126 trading days"},
        {"family": "high_low_range", "example_features": "high_low_range_21d, high_low_range_63d, high_low_range_126d", "windows": "21, 63, 126 trading days"},
        {"family": "cross_sectional_ranks", "example_features": "cs_rank_return_252d, cs_rank_avg_dollar_volume_63d", "windows": "all retained numeric headline features"},
        {"family": "market_regime", "example_features": "market_spy_return_63d, macro_yield_curve_10y_2y, vix_close", "windows": "monthly as-of signal date"},
        {"family": "fundamental_diagnostic", "example_features": "fundamental_roa_proxy, fundamental_debt_to_assets_proxy", "windows": f"diagnostic only with {REPORTING_LAG_DAYS}-day reporting lag where quarterly fields are available"},
    ]
)
save_artifact_table(feature_family_summary, "feature_family_summary.csv", description="Documented feature families, representative columns, and lookback windows.")

leakage_checks = pd.DataFrame(
    [
        {"check": "No random split", "status": "pass", "evidence": "Chronological date masks and monthly walk-forward CV are used."},
        {"check": "Forward columns excluded", "status": "pass", "evidence": str(not any(column.startswith("forward_") for column in ACTIVE_FEATURE_COLUMNS))},
        {"check": "Target columns excluded", "status": "pass", "evidence": str("target_top_quintile" not in ACTIVE_FEATURE_COLUMNS and "rank_relevance_quintile" not in ACTIVE_FEATURE_COLUMNS)},
        {"check": "Ticker identity excluded", "status": "pass", "evidence": "No symbol dummy variables are included in headline features."},
        {"check": "Fundamentals excluded from headline", "status": "pass", "evidence": str(not any(column.startswith("fundamental_") for column in ACTIVE_FEATURE_COLUMNS))},
        {"check": "Future eligibility count excluded", "status": "pass", "evidence": str("monthly_eligible_assets" not in ACTIVE_FEATURE_COLUMNS and not any(column.startswith("cs_rank_monthly_eligible_assets") for column in ACTIVE_FEATURE_COLUMNS))},
        {"check": "Public universe caveat", "status": "warning", "evidence": "Static current large-cap universe is survivorship-biased and documented in method_contract.md."},
        {"check": "Public fundamentals caveat", "status": "warning", "evidence": "Fundamentals are diagnostic only and documented in fundamental_diagnostic_caveat.md."},
    ]
)
save_artifact_table(leakage_checks, "leakage_checks.csv", description="Leakage and interpretation checklist.")
display(leakage_checks)

def population_stability_index(expected, actual, n_bins=10):
    expected = pd.Series(expected).replace([np.inf, -np.inf], np.nan).dropna()
    actual = pd.Series(actual).replace([np.inf, -np.inf], np.nan).dropna()
    if len(expected) < n_bins or len(actual) < n_bins or expected.nunique() < 2:
        return np.nan
    quantiles = np.linspace(0.0, 1.0, n_bins + 1)
    edges = np.unique(np.nanquantile(expected, quantiles))
    if len(edges) < 3:
        return np.nan
    edges[0] = -np.inf
    edges[-1] = np.inf
    expected_bins = pd.cut(expected, bins=edges, include_lowest=True)
    actual_bins = pd.cut(actual, bins=edges, include_lowest=True)
    expected_share = expected_bins.value_counts(sort=False, normalize=True).replace(0, 1e-6)
    actual_share = actual_bins.value_counts(sort=False, normalize=True).reindex(expected_share.index).fillna(1e-6).replace(0, 1e-6)
    return float(((actual_share - expected_share) * np.log(actual_share / expected_share)).sum())


drift_rows = []
for column in tqdm(ACTIVE_FEATURE_COLUMNS, desc="Computing feature drift", unit="feature"):
    if column not in selection_df.columns or column not in holdout_df.columns:
        continue
    if not pd.api.types.is_numeric_dtype(selection_df[column]):
        continue
    drift_rows.append(
        {
            "feature": column,
            "selection_missing_rate": float(selection_df[column].isna().mean()),
            "holdout_missing_rate": float(holdout_df[column].isna().mean()),
            "selection_mean": float(selection_df[column].mean(skipna=True)) if selection_df[column].notna().any() else np.nan,
            "holdout_mean": float(holdout_df[column].mean(skipna=True)) if holdout_df[column].notna().any() else np.nan,
            "psi_selection_to_holdout": population_stability_index(selection_df[column], holdout_df[column], n_bins=10),
        }
    )
feature_drift_summary = pd.DataFrame(drift_rows).sort_values("psi_selection_to_holdout", ascending=False, na_position="last")
save_artifact_table(feature_drift_summary, "feature_drift_psi_summary.csv", description="Selection-to-holdout feature drift using population stability index.")

target_stability_summary = pd.concat(
    [
        target_summary[target_summary["date"] <= pd.Timestamp(MODEL_SELECTION_END_DATE)].assign(split="model_selection"),
        target_summary[(target_summary["date"] > pd.Timestamp(MODEL_SELECTION_END_DATE)) & (target_summary["date"] <= pd.Timestamp(CALIBRATION_END_DATE))].assign(split="calibration"),
        target_summary[target_summary["date"] >= pd.Timestamp(HOLDOUT_START_DATE)].assign(split="holdout"),
    ],
    ignore_index=True,
).groupby("split").agg(
    months=("date", "count"),
    mean_rows=("rows", "mean"),
    mean_positive_rate=("positive_rate", "mean"),
    mean_forward_return=("mean_forward_return", "mean"),
    median_forward_return=("median_forward_return", "median"),
).reset_index()
save_artifact_table(target_stability_summary, "target_stability_summary.csv", description="Target and forward-return stability by chronological split.")

sector_stability_summary = (
    pd.concat(
        [
            selection_df[["sector"]].assign(split="model_selection"),
            calibration_df[["sector"]].assign(split="calibration"),
            holdout_df[["sector"]].assign(split="holdout"),
        ],
        ignore_index=True,
    )
    .groupby(["split", "sector"])
    .size()
    .rename("rows")
    .reset_index()
)
sector_stability_summary["split_share"] = sector_stability_summary["rows"] / sector_stability_summary.groupby("split")["rows"].transform("sum")
save_artifact_table(sector_stability_summary, "sector_stability_summary.csv", description="Sector row-share stability by split.")


def make_walk_forward_month_splits(frame, y, n_splits=WALK_FORWARD_CV_SPLITS, embargo_months=EMBARGO_MONTHS):
    months = pd.Series(pd.to_datetime(frame["date"]).dt.to_period("M").astype(str).to_numpy())
    unique_months = np.array(sorted(months.unique()))
    if len(unique_months) < n_splits + embargo_months + 2:
        raise RuntimeError("Not enough months for the requested walk-forward split count.")
    boundaries = np.unique(np.linspace(max(12, int(len(unique_months) * 0.35)), len(unique_months), n_splits + 1, dtype=int))
    splits = []
    rows = []
    y_values = y_to_numpy(y)
    for fold_idx, (validation_start, validation_stop) in enumerate(zip(boundaries[:-1], boundaries[1:]), start=1):
        train_stop = max(validation_start - embargo_months, 1)
        train_months = set(unique_months[:train_stop])
        validation_months = set(unique_months[validation_start:validation_stop])
        train_idx = np.where(months.isin(train_months).to_numpy())[0]
        validation_idx = np.where(months.isin(validation_months).to_numpy())[0]
        train_positive = int(y_values[train_idx].sum()) if len(train_idx) else 0
        validation_positive = int(y_values[validation_idx].sum()) if len(validation_idx) else 0
        used = len(train_idx) > 0 and len(validation_idx) > 0 and train_positive >= MIN_CV_TRAIN_POSITIVES and validation_positive >= MIN_CV_VALIDATION_POSITIVES
        rows.append(
            {
                "fold": fold_idx,
                "train_rows": len(train_idx),
                "validation_rows": len(validation_idx),
                "train_months": len(train_months),
                "validation_months": len(validation_months),
                "train_positive_rows": train_positive,
                "validation_positive_rows": validation_positive,
                "used": used,
                "train_end_month": unique_months[train_stop - 1] if train_stop else "",
                "validation_start_month": unique_months[validation_start] if validation_start < len(unique_months) else "",
                "validation_end_month": unique_months[validation_stop - 1] if validation_stop else "",
            }
        )
        if used:
            splits.append((train_idx, validation_idx))
    return splits, pd.DataFrame(rows)


cv_splits, cv_split_summary = make_walk_forward_month_splits(selection_df, y_selection)
save_artifact_table(cv_split_summary, "cv_split_summary.csv", description="Monthly walk-forward CV split summary.")
display(cv_split_summary)
if not cv_splits:
    raise RuntimeError("No usable CV folds were created for model selection.")

## 4. Model Registry, Tuning, and Evaluation Helpers

This section defines GPU-aware XGBoost wrappers, broad randomized search utilities, ranking metrics, calibration metrics, and portfolio diagnostics. Hyperparameter search samples continuous and integer distributions rather than selecting from a short hand-written list.

In [ ]:
def safe_metric(metric_fn, y_true, y_score):
    y_true_values = y_to_numpy(y_true)
    y_score_values = np.asarray(y_score, dtype=float)
    if len(np.unique(y_true_values)) < 2:
        return np.nan
    try:
        return float(metric_fn(y_true_values, y_score_values))
    except Exception:
        return np.nan


def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = pd.Series(y_to_numpy(y_true))
    y_proba = pd.Series(np.asarray(y_proba, dtype=float)).clip(0.0, 1.0)
    if y_proba.nunique(dropna=True) <= 1:
        return np.nan
    n_bins = min(n_bins, y_proba.nunique(dropna=True))
    try:
        bins = pd.qcut(y_proba, q=n_bins, duplicates="drop")
    except ValueError:
        return np.nan
    frame = pd.DataFrame({"y_true": y_true, "y_proba": y_proba, "bin": bins})
    grouped = frame.groupby("bin", observed=True)
    weights = grouped.size() / len(frame)
    observed = grouped["y_true"].mean()
    predicted = grouped["y_proba"].mean()
    return float((weights * (observed - predicted).abs()).sum())


def calibration_bin_table(y_true, y_proba, n_bins=10):
    y_true = pd.Series(y_to_numpy(y_true))
    y_proba = pd.Series(np.asarray(y_proba, dtype=float)).clip(0.0, 1.0)
    n_bins = min(n_bins, max(1, y_proba.nunique(dropna=True)))
    try:
        bins = pd.qcut(y_proba, q=n_bins, duplicates="drop")
    except ValueError:
        bins = pd.cut(y_proba, bins=n_bins, include_lowest=True, duplicates="drop")
    frame = pd.DataFrame({"y_true": y_true, "y_proba": y_proba, "bin": bins})
    return (
        frame.groupby("bin", observed=True)
        .agg(rows=("y_true", "size"), observed_rate=("y_true", "mean"), mean_predicted_probability=("y_proba", "mean"), score_min=("y_proba", "min"), score_max=("y_proba", "max"))
        .reset_index()
    )


def positive_class_proba(model, X):
    proba = model.predict_proba(X)
    proba = cp.asnumpy(proba) if hasattr(proba, "get") else np.asarray(proba)
    if proba.ndim == 1:
        return proba.astype(float)
    if proba.shape[1] == 1:
        return proba[:, 0].astype(float)
    return proba[:, 1].astype(float)


def predict_proba_in_chunks(model, X, chunk_size=PREDICTION_CHUNK_SIZE, desc="Predicting probabilities"):
    if len(X) <= chunk_size:
        return positive_class_proba(model, X)
    parts = []
    for start in tqdm(range(0, len(X), chunk_size), desc=desc, unit="chunk", leave=False):
        stop = min(start + chunk_size, len(X))
        parts.append(positive_class_proba(model, take_rows(X, np.arange(start, stop))))
    return np.concatenate(parts)


def predict_scores_in_chunks(model, X, chunk_size=PREDICTION_CHUNK_SIZE, desc="Predicting scores"):
    if len(X) <= chunk_size:
        score = model.predict(X)
        return cp.asnumpy(score) if hasattr(score, "get") else np.asarray(score, dtype=float)
    parts = []
    for start in tqdm(range(0, len(X), chunk_size), desc=desc, unit="chunk", leave=False):
        stop = min(start + chunk_size, len(X))
        score = model.predict(take_rows(X, np.arange(start, stop)))
        score = cp.asnumpy(score) if hasattr(score, "get") else np.asarray(score, dtype=float)
        parts.append(score)
    return np.concatenate(parts)


class GPUXGBClassifier(XGBClassifier):
    def fit(self, X, y, **kwargs):
        fit_kwargs = dict(kwargs)
        if XGBOOST_DEVICE == "cuda":
            X_fit = to_cupy_float32(X)
            y_fit = cp.asarray(y_to_numpy(y))
            if fit_kwargs.get("eval_set") is not None:
                fit_kwargs["eval_set"] = [(to_cupy_float32(eval_X), cp.asarray(y_to_numpy(eval_y))) for eval_X, eval_y in fit_kwargs["eval_set"]]
        else:
            X_fit = to_numpy_float32(X)
            y_fit = y_to_numpy(y)
            if fit_kwargs.get("eval_set") is not None:
                fit_kwargs["eval_set"] = [(to_numpy_float32(eval_X), y_to_numpy(eval_y)) for eval_X, eval_y in fit_kwargs["eval_set"]]
        return super().fit(X_fit, y_fit, **fit_kwargs)

    def predict_proba(self, X, **kwargs):
        X_predict = to_cupy_float32(X) if XGBOOST_DEVICE == "cuda" else to_numpy_float32(X)
        proba = super().predict_proba(X_predict, **kwargs)
        return cp.asnumpy(proba) if hasattr(proba, "get") else np.asarray(proba)

    def predict(self, X, **kwargs):
        X_predict = to_cupy_float32(X) if XGBOOST_DEVICE == "cuda" else to_numpy_float32(X)
        prediction = super().predict(X_predict, **kwargs)
        return cp.asnumpy(prediction) if hasattr(prediction, "get") else np.asarray(prediction)


class GPUXGBRanker(XGBRanker):
    def fit(self, X, y, **kwargs):
        fit_kwargs = dict(kwargs)
        if XGBOOST_DEVICE == "cuda":
            X_fit = to_cupy_float32(X)
            y_fit = cp.asarray(np.asarray(y, dtype=np.float32))
            if fit_kwargs.get("eval_set") is not None:
                fit_kwargs["eval_set"] = [(to_cupy_float32(eval_X), cp.asarray(np.asarray(eval_y, dtype=np.float32))) for eval_X, eval_y in fit_kwargs["eval_set"]]
        else:
            X_fit = to_numpy_float32(X)
            y_fit = np.asarray(y, dtype=np.float32)
            if fit_kwargs.get("eval_set") is not None:
                fit_kwargs["eval_set"] = [(to_numpy_float32(eval_X), np.asarray(eval_y, dtype=np.float32)) for eval_X, eval_y in fit_kwargs["eval_set"]]
        return super().fit(X_fit, y_fit, **fit_kwargs)

    def predict(self, X, **kwargs):
        X_predict = to_cupy_float32(X) if XGBOOST_DEVICE == "cuda" else to_numpy_float32(X)
        prediction = super().predict(X_predict, **kwargs)
        return cp.asnumpy(prediction) if hasattr(prediction, "get") else np.asarray(prediction)


def make_xgb_classifier(n_estimators=XGBOOST_FINAL_N_ESTIMATORS, y_for_weight=None):
    weight_target = y_selection if y_for_weight is None else y_for_weight
    positive = max(int(pd.Series(weight_target).sum()), 1)
    negative = max(int(len(weight_target) - pd.Series(weight_target).sum()), 1)
    return GPUXGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method=XGBOOST_TREE_METHOD,
        device=XGBOOST_DEVICE,
        random_state=SEED,
        n_jobs=1,
        verbosity=0,
        n_estimators=n_estimators,
        scale_pos_weight=negative / positive,
    )


def make_xgb_ranker(n_estimators=XGBOOST_FINAL_N_ESTIMATORS):
    return GPUXGBRanker(
        objective="rank:ndcg",
        eval_metric="ndcg@10",
        tree_method=XGBOOST_TREE_METHOD,
        device=XGBOOST_DEVICE,
        random_state=SEED,
        n_jobs=1,
        verbosity=0,
        n_estimators=n_estimators,
    )


CLASSIFIER_PARAM_DISTRIBUTIONS = {
    "max_depth": randint(2, 9),
    "learning_rate": loguniform(0.005, 0.12),
    "subsample": uniform(0.60, 0.38),
    "colsample_bytree": uniform(0.45, 0.50),
    "min_child_weight": loguniform(0.5, 80.0),
    "gamma": loguniform(1e-4, 10.0),
    "reg_alpha": loguniform(1e-5, 25.0),
    "reg_lambda": loguniform(0.5, 150.0),
    "max_delta_step": randint(0, 8),
}

RANKER_PARAM_DISTRIBUTIONS = {
    "max_depth": randint(2, 8),
    "learning_rate": loguniform(0.005, 0.10),
    "subsample": uniform(0.60, 0.38),
    "colsample_bytree": uniform(0.45, 0.50),
    "min_child_weight": loguniform(0.5, 80.0),
    "gamma": loguniform(1e-4, 15.0),
    "reg_alpha": loguniform(1e-5, 25.0),
    "reg_lambda": loguniform(0.5, 150.0),
}


def group_sizes_by_month(frame):
    return frame.groupby("date", sort=True).size().astype(int).to_numpy()


def monthly_rank_ic(frame, score_column="score", target_column="forward_1m_open_return"):
    rows = []
    for date, group in frame.dropna(subset=[score_column, target_column]).groupby("date", sort=True):
        if group[score_column].nunique() < 2 or group[target_column].nunique() < 2:
            rank_ic = np.nan
        else:
            rank_ic = group[score_column].corr(group[target_column], method="spearman")
        rows.append({"date": date, "rank_ic": rank_ic, "rows": len(group)})
    return pd.DataFrame(rows)


def ranking_diagnostics(prediction_frame, top_quantile=TOP_QUANTILE, bottom_quantile=LONG_SHORT_BOTTOM_QUANTILE):
    rows = []
    for date, group in prediction_frame.dropna(subset=["score", "forward_1m_open_return"]).groupby("date", sort=True):
        group = group.sort_values(["score", "symbol"], ascending=[False, True]).copy()
        n = len(group)
        if n < 2:
            continue
        top_k = max(1, int(math.ceil(n * top_quantile))) if TOP_K_PORTFOLIO is None else min(TOP_K_PORTFOLIO, n)
        bottom_k = max(1, int(math.ceil(n * bottom_quantile)))
        top = group.head(top_k)
        bottom = group.tail(bottom_k)
        rank_ic = group["score"].corr(group["forward_1m_open_return"], method="spearman") if group["score"].nunique() > 1 and group["forward_1m_open_return"].nunique() > 1 else np.nan
        positives = group["target_top_quintile"].astype(int)
        average_precision = average_precision_score(positives, group["score"]) if positives.nunique() > 1 else np.nan
        rows.append(
            {
                "date": date,
                "rows": n,
                "top_k": top_k,
                "bottom_k": bottom_k,
                "rank_ic": rank_ic,
                "average_precision": average_precision,
                "precision_at_top_k": float(top["target_top_quintile"].mean()),
                "top_mean_return": float(top["forward_1m_open_return"].mean()),
                "bottom_mean_return": float(bottom["forward_1m_open_return"].mean()),
                "universe_mean_return": float(group["forward_1m_open_return"].mean()),
                "top_minus_bottom_return": float(top["forward_1m_open_return"].mean() - bottom["forward_1m_open_return"].mean()),
                "top_excess_return": float(top["forward_1m_open_return"].mean() - group["forward_1m_open_return"].mean()),
            }
        )
    return pd.DataFrame(rows)


def annualized_summary(monthly_returns, name, transaction_cost_bps=0.0):
    returns_series = pd.Series(monthly_returns).dropna().astype(float)
    if returns_series.empty:
        return {"name": name, "months": 0, "total_return": np.nan, "cagr": np.nan, "annualized_vol": np.nan, "sharpe": np.nan, "max_drawdown": np.nan, "mean_monthly_return": np.nan}
    equity = (1.0 + returns_series).cumprod()
    years = max(len(returns_series) / 12.0, 1.0 / 12.0)
    cagr = float(equity.iloc[-1] ** (1.0 / years) - 1.0) if equity.iloc[-1] > 0 else np.nan
    annualized_vol = float(returns_series.std(ddof=1) * math.sqrt(12.0)) if len(returns_series) > 1 else np.nan
    sharpe = float((returns_series.mean() * 12.0) / annualized_vol) if annualized_vol and annualized_vol > 0 else np.nan
    drawdown = equity / equity.cummax() - 1.0
    return {
        "name": name,
        "months": int(len(returns_series)),
        "transaction_cost_bps": transaction_cost_bps,
        "total_return": float(equity.iloc[-1] - 1.0),
        "cagr": cagr,
        "annualized_vol": annualized_vol,
        "sharpe": sharpe,
        "max_drawdown": float(drawdown.min()),
        "mean_monthly_return": float(returns_series.mean()),
    }


def score_to_portfolio_returns(prediction_frame, model_name, transaction_cost_bps=TRANSACTION_COST_BPS):
    previous_long_weights = None
    previous_ls_weights = None
    rows = []
    cost_rate = transaction_cost_bps / 10000.0
    for date, group in prediction_frame.dropna(subset=["score", "forward_1m_open_return"]).groupby("date", sort=True):
        group = group.sort_values(["score", "symbol"], ascending=[False, True]).copy()
        n = len(group)
        top_k = max(1, int(math.ceil(n * TOP_QUANTILE))) if TOP_K_PORTFOLIO is None else min(TOP_K_PORTFOLIO, n)
        bottom_k = max(1, int(math.ceil(n * LONG_SHORT_BOTTOM_QUANTILE)))
        top_symbols = group.head(top_k)["symbol"].tolist()
        bottom_symbols = group.tail(bottom_k)["symbol"].tolist()
        long_weights = pd.Series(0.0, index=group["symbol"])
        long_weights.loc[top_symbols] = 1.0 / top_k
        ls_weights = pd.Series(0.0, index=group["symbol"])
        ls_weights.loc[top_symbols] = 0.5 / top_k
        ls_weights.loc[bottom_symbols] = -0.5 / bottom_k
        returns_by_symbol = group.set_index("symbol")["forward_1m_open_return"]
        long_turnover = float(long_weights.abs().sum()) if previous_long_weights is None else float((long_weights - previous_long_weights.reindex(long_weights.index).fillna(0.0)).abs().sum())
        ls_turnover = float(ls_weights.abs().sum()) if previous_ls_weights is None else float((ls_weights - previous_ls_weights.reindex(ls_weights.index).fillna(0.0)).abs().sum())
        long_gross_return = float((long_weights * returns_by_symbol).sum())
        ls_gross_return = float((ls_weights * returns_by_symbol).sum())
        rows.append(
            {
                "date": date,
                "Model": model_name,
                "long_top_return_gross": long_gross_return,
                "long_top_turnover": long_turnover,
                "long_top_return_net": long_gross_return - cost_rate * long_turnover,
                "long_short_return_gross": ls_gross_return,
                "long_short_turnover": ls_turnover,
                "long_short_return_net": ls_gross_return - cost_rate * ls_turnover,
                "top_k": top_k,
                "bottom_k": bottom_k,
                "avg_top_dollar_volume_63d": float(group.head(top_k)["avg_dollar_volume_63d"].mean()) if "avg_dollar_volume_63d" in group else np.nan,
                "avg_top_high_low_range_21d": float(group.head(top_k)["high_low_range_21d"].mean()) if "high_low_range_21d" in group else np.nan,
                "top_sector_count": int(group.head(top_k)["sector"].nunique()) if "sector" in group else np.nan,
                "estimated_long_top_trade_notional_usd": MARKET_IMPACT_NOTIONAL_USD * long_turnover,
                "estimated_long_top_avg_adv_participation": (MARKET_IMPACT_NOTIONAL_USD * long_turnover / max(top_k, 1)) / group.head(top_k)["avg_dollar_volume_63d"].replace(0, np.nan).mean() if "avg_dollar_volume_63d" in group else np.nan,
                "estimated_long_short_trade_notional_usd": MARKET_IMPACT_NOTIONAL_USD * ls_turnover,
                "estimated_long_short_avg_adv_participation": (MARKET_IMPACT_NOTIONAL_USD * ls_turnover / max(top_k + bottom_k, 1)) / group[["symbol", "avg_dollar_volume_63d"]].set_index("symbol").reindex(top_symbols + bottom_symbols)["avg_dollar_volume_63d"].replace(0, np.nan).mean() if "avg_dollar_volume_63d" in group else np.nan,
            }
        )
        previous_long_weights = long_weights.copy()
        previous_ls_weights = ls_weights.copy()
    return pd.DataFrame(rows)


def score_to_sector_neutral_portfolio_returns(prediction_frame, model_name, transaction_cost_bps=TRANSACTION_COST_BPS):
    previous_long_weights = None
    previous_ls_weights = None
    rows = []
    cost_rate = transaction_cost_bps / 10000.0
    for date, group in prediction_frame.dropna(subset=["score", "forward_1m_open_return", "sector"]).groupby("date", sort=True):
        selected_long = []
        selected_short = []
        for _, sector_group in group.groupby("sector", sort=True):
            sector_group = sector_group.sort_values(["score", "symbol"], ascending=[False, True])
            k = max(1, int(math.ceil(len(sector_group) * TOP_QUANTILE)))
            selected_long.extend(sector_group.head(k)["symbol"].tolist())
            selected_short.extend(sector_group.tail(k)["symbol"].tolist())
        selected_long = sorted(set(selected_long))
        selected_short = sorted(set(selected_short))
        if not selected_long or not selected_short:
            continue
        all_symbols = group["symbol"].astype(str).tolist()
        long_weights = pd.Series(0.0, index=all_symbols)
        long_weights.loc[selected_long] = 1.0 / len(selected_long)
        ls_weights = pd.Series(0.0, index=all_symbols)
        ls_weights.loc[selected_long] = 0.5 / len(selected_long)
        ls_weights.loc[selected_short] = -0.5 / len(selected_short)
        returns_by_symbol = group.set_index("symbol")["forward_1m_open_return"]
        long_turnover = float(long_weights.abs().sum()) if previous_long_weights is None else float((long_weights - previous_long_weights.reindex(long_weights.index).fillna(0.0)).abs().sum())
        ls_turnover = float(ls_weights.abs().sum()) if previous_ls_weights is None else float((ls_weights - previous_ls_weights.reindex(ls_weights.index).fillna(0.0)).abs().sum())
        long_gross_return = float((long_weights * returns_by_symbol).sum())
        ls_gross_return = float((ls_weights * returns_by_symbol).sum())
        selected_adv = group.set_index("symbol").reindex(selected_long)["avg_dollar_volume_63d"] if "avg_dollar_volume_63d" in group else pd.Series(dtype=float)
        selected_ls_adv = group.set_index("symbol").reindex(selected_long + selected_short)["avg_dollar_volume_63d"] if "avg_dollar_volume_63d" in group else pd.Series(dtype=float)
        rows.append(
            {
                "date": date,
                "Model": f"{model_name} sector-neutral",
                "Source Model": model_name,
                "long_top_return_gross": long_gross_return,
                "long_top_turnover": long_turnover,
                "long_top_return_net": long_gross_return - cost_rate * long_turnover,
                "long_short_return_gross": ls_gross_return,
                "long_short_turnover": ls_turnover,
                "long_short_return_net": ls_gross_return - cost_rate * ls_turnover,
                "top_k": len(selected_long),
                "bottom_k": len(selected_short),
                "avg_top_dollar_volume_63d": float(selected_adv.replace(0, np.nan).mean()) if len(selected_adv) else np.nan,
                "avg_top_high_low_range_21d": float(group.set_index("symbol").reindex(selected_long)["high_low_range_21d"].mean()) if "high_low_range_21d" in group else np.nan,
                "top_sector_count": int(group.loc[group["symbol"].isin(selected_long), "sector"].nunique()),
                "estimated_long_top_trade_notional_usd": MARKET_IMPACT_NOTIONAL_USD * long_turnover,
                "estimated_long_top_avg_adv_participation": (MARKET_IMPACT_NOTIONAL_USD * long_turnover / max(len(selected_long), 1)) / selected_adv.replace(0, np.nan).mean() if len(selected_adv) else np.nan,
                "estimated_long_short_trade_notional_usd": MARKET_IMPACT_NOTIONAL_USD * ls_turnover,
                "estimated_long_short_avg_adv_participation": (MARKET_IMPACT_NOTIONAL_USD * ls_turnover / max(len(selected_long) + len(selected_short), 1)) / selected_ls_adv.replace(0, np.nan).mean() if len(selected_ls_adv) else np.nan,
            }
        )
        previous_long_weights = long_weights.copy()
        previous_ls_weights = ls_weights.copy()
    return pd.DataFrame(rows)


def equal_weight_universe_returns(frame, transaction_cost_bps=TRANSACTION_COST_BPS):
    previous_weights = None
    rows = []
    cost_rate = transaction_cost_bps / 10000.0
    for date, group in frame.dropna(subset=["forward_1m_open_return"]).groupby("date", sort=True):
        symbols = group["symbol"].astype(str).tolist()
        weights = pd.Series(1.0 / len(symbols), index=symbols)
        returns_by_symbol = group.set_index("symbol")["forward_1m_open_return"]
        turnover = float(weights.abs().sum()) if previous_weights is None else float((weights - previous_weights.reindex(weights.index).fillna(0.0)).abs().sum())
        gross_return = float((weights * returns_by_symbol).sum())
        rows.append(
            {
                "date": date,
                "Model": "Benchmark[Equal-weight universe]",
                "long_top_return_gross": gross_return,
                "long_top_turnover": turnover,
                "long_top_return_net": gross_return - cost_rate * turnover,
                "long_short_return_gross": np.nan,
                "long_short_turnover": np.nan,
                "long_short_return_net": np.nan,
                "top_k": len(symbols),
                "bottom_k": 0,
                "avg_top_dollar_volume_63d": float(group["avg_dollar_volume_63d"].mean()) if "avg_dollar_volume_63d" in group else np.nan,
                "avg_top_high_low_range_21d": float(group["high_low_range_21d"].mean()) if "high_low_range_21d" in group else np.nan,
                "top_sector_count": int(group["sector"].nunique()) if "sector" in group else np.nan,
                "estimated_long_top_trade_notional_usd": MARKET_IMPACT_NOTIONAL_USD * turnover,
                "estimated_long_top_avg_adv_participation": (MARKET_IMPACT_NOTIONAL_USD * turnover / max(len(symbols), 1)) / group["avg_dollar_volume_63d"].replace(0, np.nan).mean() if "avg_dollar_volume_63d" in group else np.nan,
                "estimated_long_short_trade_notional_usd": np.nan,
                "estimated_long_short_avg_adv_participation": np.nan,
            }
        )
        previous_weights = weights.copy()
    return pd.DataFrame(rows)


def metric_row(model_name, base_model, family, score_type, evaluation_window, y_true, y_score, fit_seconds=0.0, predict_seconds=0.0, tuning_score=np.nan, best_params="", notes=""):
    y_true_values = y_to_numpy(y_true)
    y_score_values = np.asarray(y_score, dtype=float)
    return {
        "Model": model_name,
        "Base Model": base_model,
        "Family": family,
        "Score Type": score_type,
        "Evaluation Window": evaluation_window,
        "Rows": len(y_true_values),
        "Positive Rows": int(y_true_values.sum()),
        "Positive Rate": float(y_true_values.mean()) if len(y_true_values) else np.nan,
        "Average Precision": safe_metric(average_precision_score, y_true_values, y_score_values),
        "ROC AUC": safe_metric(roc_auc_score, y_true_values, y_score_values),
        "Brier Score": brier_score_loss(y_true_values, np.clip(y_score_values, 0.0, 1.0)) if score_type == "probability" and len(np.unique(y_true_values)) > 1 else np.nan,
        "Log Loss": log_loss(y_true_values, np.clip(y_score_values, 1e-6, 1.0 - 1e-6), labels=[0, 1]) if score_type == "probability" and len(np.unique(y_true_values)) > 1 else np.nan,
        "ECE Quantile 10": expected_calibration_error(y_true_values, y_score_values, 10) if score_type == "probability" else np.nan,
        "Fit Seconds": fit_seconds,
        "Predict Seconds": predict_seconds,
        "Workflow Seconds": fit_seconds + predict_seconds,
        "CV/Validation Score": tuning_score,
        "Best Params": best_params,
        "Notes": notes,
    }


def build_prediction_frame(model_name, frame, scores, evaluation_window, score_type):
    columns = [
        "date", "month", "symbol", "sector", "industry", "target_top_quintile", "rank_relevance_quintile",
        "forward_1m_open_return", "forward_1m_rank", "forward_1m_pct_rank", "monthly_eligible_assets",
        "avg_dollar_volume_63d", "high_low_range_21d",
    ]
    available_columns = [column for column in columns if column in frame.columns]
    out = frame[available_columns].copy()
    out["Model"] = model_name
    out["Evaluation Window"] = evaluation_window
    out["Score Type"] = score_type
    out["score"] = np.asarray(scores, dtype=np.float32)
    return out


def add_predictions(model_name, base_model, family, score_type, eval_payloads, fit_seconds=0.0, tuning_score=np.nan, best_params="", notes=""):
    for evaluation_window, frame, y_true, scores, predict_seconds in eval_payloads:
        scores = np.asarray(scores, dtype=np.float32)
        prediction_frames.append(build_prediction_frame(model_name, frame, scores, evaluation_window, score_type))
        model_rows.append(metric_row(model_name, base_model, family, score_type, evaluation_window, y_true, scores, fit_seconds, predict_seconds, tuning_score, best_params, notes))


class ProgressRandomizedSearchCV:
    def __init__(self, estimator, param_distributions, n_iter, cv_splits, random_state=SEED, label="model search", metric="average_precision"):
        self.estimator = estimator
        self.param_distributions = param_distributions
        self.n_iter = int(n_iter)
        self.cv_splits = list(cv_splits)
        self.random_state = random_state
        self.label = label
        self.metric = metric

    def cv_results_frame(self):
        rows = []
        for row in getattr(self, "trial_rows_", []):
            cleaned = dict(row)
            cleaned["params"] = artifact_json(cleaned.get("params", {}))
            cleaned["fold_scores"] = artifact_json(cleaned.get("fold_scores", []))
            cleaned["fold_errors"] = artifact_json(cleaned.get("fold_errors", []))
            cleaned["fold_best_iterations"] = artifact_json(cleaned.get("fold_best_iterations", []))
            rows.append(cleaned)
        return pd.DataFrame(rows)

    def selected_n_estimators(self, default_n_estimators=XGBOOST_FINAL_N_ESTIMATORS):
        values = []
        for row in getattr(self, "trial_rows_", []):
            if row.get("params") != getattr(self, "best_params_", None):
                continue
            values.extend([value for value in row.get("fold_best_iterations", []) if pd.notna(value) and value > 0])
        if not values:
            return int(default_n_estimators)
        # A modest buffer around the median fold best iteration keeps the final fit efficient without using holdout labels.
        return int(max(50, min(default_n_estimators, math.ceil(float(np.nanmedian(values)) * 1.15))))

    @staticmethod
    def _best_iteration_or_nan(estimator):
        best_iteration = getattr(estimator, "best_iteration", None)
        if best_iteration is None:
            best_iteration = getattr(estimator, "best_iteration_", None)
        if best_iteration is None:
            return np.nan
        return int(best_iteration) + 1

    def fit_classifier(self, X, y):
        parameter_draws = list(ParameterSampler(self.param_distributions, n_iter=self.n_iter, random_state=self.random_state))
        best_score = -np.inf
        best_params = None
        trial_rows = []
        progress = tqdm(parameter_draws, desc=f"Tuning {self.label}", unit="trial")
        for trial_idx, params in enumerate(progress, start=1):
            fold_scores = []
            fold_errors = []
            fold_best_iterations = []
            trial_start = time.time()
            for fold_idx, (train_idx, validation_idx) in enumerate(self.cv_splits, start=1):
                estimator = clone(self.estimator)
                estimator.set_params(**params, early_stopping_rounds=XGBOOST_SEARCH_EARLY_STOPPING_ROUNDS)
                try:
                    estimator.fit(take_rows(X, train_idx), take_rows(y, train_idx), eval_set=[(take_rows(X, validation_idx), take_rows(y, validation_idx))], verbose=False)
                    validation_score = positive_class_proba(estimator, take_rows(X, validation_idx))
                    fold_score = safe_metric(average_precision_score, take_rows(y, validation_idx), validation_score)
                    fold_best_iterations.append(self._best_iteration_or_nan(estimator))
                except Exception as exc:
                    fold_score = np.nan
                    fold_errors.append(f"fold_{fold_idx}: {short_error(exc)}")
                    fold_best_iterations.append(np.nan)
                fold_scores.append(fold_score)
                del estimator
                cleanup_runtime_memory("classifier_cv_fold")
            fold_scores_array = np.asarray(fold_scores, dtype=float)
            best_iterations_array = np.asarray(fold_best_iterations, dtype=float)
            mean_score = float(np.nanmean(fold_scores_array)) if np.isfinite(fold_scores_array).any() else np.nan
            std_score = float(np.nanstd(fold_scores_array)) if np.isfinite(fold_scores_array).any() else np.nan
            mean_best_iteration = float(np.nanmean(best_iterations_array)) if np.isfinite(best_iterations_array).any() else np.nan
            trial_seconds = time.time() - trial_start
            trial_rows.append({"trial": trial_idx, "mean_score": mean_score, "std_score": std_score, "trial_seconds": trial_seconds, "mean_best_iteration": mean_best_iteration, "fold_scores": fold_scores, "fold_errors": fold_errors, "fold_best_iterations": fold_best_iterations, "params": params})
            progress.set_postfix(best=f"{best_score:.4f}" if np.isfinite(best_score) else "nan", current=f"{mean_score:.4f}" if np.isfinite(mean_score) else "nan", best_iter=f"{mean_best_iteration:.0f}" if np.isfinite(mean_best_iteration) else "nan")
            if np.isfinite(mean_score) and mean_score > best_score:
                best_score = mean_score
                best_params = params
        self.trial_rows_ = trial_rows
        self.best_score_ = best_score
        self.best_params_ = best_params or {}
        self.best_n_estimators_ = self.selected_n_estimators()
        return self

    def fit_ranker(self, X, y, frame):
        parameter_draws = list(ParameterSampler(self.param_distributions, n_iter=self.n_iter, random_state=self.random_state))
        best_score = -np.inf
        best_params = None
        trial_rows = []
        progress = tqdm(parameter_draws, desc=f"Tuning {self.label}", unit="trial")
        for trial_idx, params in enumerate(progress, start=1):
            fold_scores = []
            fold_errors = []
            fold_best_iterations = []
            trial_start = time.time()
            for fold_idx, (train_idx, validation_idx) in enumerate(self.cv_splits, start=1):
                estimator = clone(self.estimator)
                estimator.set_params(**params, early_stopping_rounds=XGBOOST_SEARCH_EARLY_STOPPING_ROUNDS)
                train_frame = take_rows(frame, train_idx).sort_values(["date", "symbol"])
                validation_frame = take_rows(frame, validation_idx).sort_values(["date", "symbol"])
                train_order = train_frame.index
                validation_order = validation_frame.index
                train_group = group_sizes_by_month(train_frame)
                validation_group = group_sizes_by_month(validation_frame)
                try:
                    estimator.fit(
                        X.loc[train_order],
                        y.loc[train_order],
                        group=train_group,
                        eval_set=[(X.loc[validation_order], y.loc[validation_order])],
                        eval_group=[validation_group],
                        verbose=False,
                    )
                    validation_score = estimator.predict(X.loc[validation_order])
                    validation_predictions = validation_frame[["date", "symbol", "forward_1m_open_return", "target_top_quintile"]].copy()
                    validation_predictions["score"] = validation_score
                    diagnostics = ranking_diagnostics(validation_predictions)
                    fold_score = float(diagnostics["rank_ic"].mean()) if len(diagnostics) else np.nan
                    fold_best_iterations.append(self._best_iteration_or_nan(estimator))
                except Exception as exc:
                    fold_score = np.nan
                    fold_errors.append(f"fold_{fold_idx}: {short_error(exc)}")
                    fold_best_iterations.append(np.nan)
                fold_scores.append(fold_score)
                del estimator
                cleanup_runtime_memory("ranker_cv_fold")
            fold_scores_array = np.asarray(fold_scores, dtype=float)
            best_iterations_array = np.asarray(fold_best_iterations, dtype=float)
            mean_score = float(np.nanmean(fold_scores_array)) if np.isfinite(fold_scores_array).any() else np.nan
            std_score = float(np.nanstd(fold_scores_array)) if np.isfinite(fold_scores_array).any() else np.nan
            mean_best_iteration = float(np.nanmean(best_iterations_array)) if np.isfinite(best_iterations_array).any() else np.nan
            trial_seconds = time.time() - trial_start
            trial_rows.append({"trial": trial_idx, "mean_score": mean_score, "std_score": std_score, "trial_seconds": trial_seconds, "mean_best_iteration": mean_best_iteration, "fold_scores": fold_scores, "fold_errors": fold_errors, "fold_best_iterations": fold_best_iterations, "params": params})
            progress.set_postfix(best=f"{best_score:.4f}" if np.isfinite(best_score) else "nan", current=f"{mean_score:.4f}" if np.isfinite(mean_score) else "nan", best_iter=f"{mean_best_iteration:.0f}" if np.isfinite(mean_best_iteration) else "nan")
            if np.isfinite(mean_score) and mean_score > best_score:
                best_score = mean_score
                best_params = params
        self.trial_rows_ = trial_rows
        self.best_score_ = best_score
        self.best_params_ = best_params or {}
        self.best_n_estimators_ = self.selected_n_estimators()
        return self


def make_prefit_calibrator(base_model, X_cal, y_cal, method="sigmoid"):
    try:
        calibrated = CalibratedClassifierCV(estimator=base_model, method=method, cv="prefit")
    except TypeError:
        calibrated = CalibratedClassifierCV(base_estimator=base_model, method=method, cv="prefit")
    calibrated.fit(X_cal, y_cal)
    return calibrated

## 5. Run Baselines, GPU XGBoost, TabPFN, and TabICL

This section evaluates deterministic ranking rules, tunes GPU XGBoost models with chronological folds, fits final models, and optionally scores the calibration and holdout windows with direct TabPFN and TabICL. Model outputs are saved as full prediction tables rather than relying on notebook display output.

In [ ]:
# Deterministic baseline scores.
def add_rule_score(model_name, score_column, ascending=False):
    for evaluation_window, frame, y_true in [("calibration", calibration_df, y_calibration), ("holdout", holdout_df, y_holdout)]:
        if score_column not in frame.columns:
            model_errors.append({"model": model_name, "stage": "rule_score", "error": f"missing score column {score_column}"})
            return
        score = frame[score_column].astype(float).to_numpy()
        if ascending:
            score = -score
        add_predictions(model_name, "Rule", "deterministic_rule", "raw_score", [(evaluation_window, frame, y_true, score, 0.0)], notes="Deterministic finance baseline; score is not a calibrated probability.")


prior_score = np.repeat(float(y_preholdout.mean()), len(holdout_df))
add_predictions("Reference[Equal-score prior]", "Dummy", "baseline", "probability", [("holdout", holdout_df, y_holdout, prior_score, 0.0)], notes="Constant equal-score reference from pre-holdout label frequency.")
random_rng = np.random.default_rng(SEED)
for evaluation_window, frame, y_true in [("calibration", calibration_df, y_calibration), ("holdout", holdout_df, y_holdout)]:
    random_scores = random_rng.random(len(frame))
    add_predictions("Reference[Seeded random score]", "Random", "baseline", "raw_score", [(evaluation_window, frame, y_true, random_scores, 0.0)], notes="Seeded random ranking reference for scale; not a finance model.")
add_rule_score("Rule[12M momentum]", "return_252d", ascending=False)
add_rule_score("Rule[6M momentum]", "return_126d", ascending=False)
add_rule_score("Rule[1M reversal]", "reversal_21d", ascending=False)
add_rule_score("Rule[Low volatility]", "rv_63d", ascending=True)
add_rule_score("Rule[Liquidity tilt]", "avg_dollar_volume_63d", ascending=False)

if RUN_CPU_LOGISTIC_BASELINE:
    start = time.time()
    logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=1))
    logistic.fit(X_preholdout, y_preholdout)
    fit_seconds = time.time() - start
    cal_start = time.time()
    cal_score = logistic.predict_proba(X_calibration)[:, 1]
    cal_seconds = time.time() - cal_start
    holdout_start = time.time()
    holdout_score = logistic.predict_proba(X_holdout)[:, 1]
    holdout_seconds = time.time() - holdout_start
    add_predictions("Logistic Regression[CPU optional]", "Logistic Regression", "cpu_optional", "probability", [("calibration", calibration_df, y_calibration, cal_score, cal_seconds), ("holdout", holdout_df, y_holdout, holdout_score, holdout_seconds)], fit_seconds=fit_seconds, notes="Optional CPU baseline; disabled by default.")

# GPU XGBoost classifier.
xgb_classifier = None
if RUN_XGBOOST_CLASSIFIER:
    try:
        classifier_search = ProgressRandomizedSearchCV(
            make_xgb_classifier(),
            CLASSIFIER_PARAM_DISTRIBUTIONS,
            XGBOOST_CLASSIFIER_TUNING_ITERATIONS,
            cv_splits,
            label="GPU XGBoost classifier",
        )
        classifier_search.fit_classifier(X_selection, y_selection)
        classifier_trials = classifier_search.cv_results_frame()
        save_artifact_table(classifier_trials, "xgboost_classifier_tuning_trials.csv", description="GPU XGBoost classifier randomized-search trial history, including fold scores, fold errors, and trial timing.")
        if not np.isfinite(classifier_search.best_score_):
            raise RuntimeError("GPU XGBoost classifier search produced no finite chronological CV score; inspect xgboost_classifier_tuning_trials.csv before fitting a final model.")
        classifier_selected_params = dict(classifier_search.best_params_)
        classifier_selected_params["n_estimators"] = classifier_search.best_n_estimators_
        save_artifact_table(pd.DataFrame([classifier_selected_params]), "xgboost_classifier_selected_params.csv", description="Selected GPU XGBoost classifier hyperparameters from chronological randomized search, including CV-derived effective estimator count.")

        xgb_classifier_selection = make_xgb_classifier()
        xgb_classifier_selection.set_params(**classifier_selected_params)
        start = time.time()
        xgb_classifier_selection.fit(X_selection, y_selection, verbose=False)
        selection_fit_seconds = time.time() - start
        start = time.time()
        cal_score = predict_proba_in_chunks(xgb_classifier_selection, X_calibration, desc="Predicting XGBoost classifier calibration")
        cal_seconds = time.time() - start

        xgb_classifier = make_xgb_classifier(y_for_weight=y_preholdout)
        xgb_classifier.set_params(**classifier_selected_params)
        start = time.time()
        xgb_classifier.fit(X_preholdout, y_preholdout, verbose=False)
        preholdout_fit_seconds = time.time() - start
        start = time.time()
        holdout_score = predict_proba_in_chunks(xgb_classifier, X_holdout, desc="Predicting XGBoost classifier holdout")
        holdout_seconds = time.time() - start

        add_predictions(
            "XGBoost[GPU top-quintile classifier]",
            "XGBClassifier",
            "gpu_gradient_boosting",
            "probability",
            [("calibration", calibration_df, y_calibration, cal_score, cal_seconds), ("holdout", holdout_df, y_holdout, holdout_score, holdout_seconds)],
            fit_seconds=selection_fit_seconds + preholdout_fit_seconds,
            tuning_score=classifier_search.best_score_,
            best_params=artifact_json(classifier_selected_params),
            notes="Broad randomized search with monthly chronological folds. Calibration scores use a model fitted only on model-selection rows; holdout scores use a separately refit pre-holdout model.",
        )

        calibrated = make_prefit_calibrator(xgb_classifier_selection, X_calibration, y_calibration, method="sigmoid")
        calibrated_holdout_score = predict_proba_in_chunks(calibrated, X_holdout, desc="Predicting calibrated XGBoost holdout")
        add_predictions(
            "XGBoost[GPU top-quintile classifier calibrated]",
            "XGBClassifier+sigmoid",
            "gpu_gradient_boosting",
            "probability",
            [("holdout", holdout_df, y_holdout, calibrated_holdout_score, 0.0)],
            fit_seconds=selection_fit_seconds,
            tuning_score=classifier_search.best_score_,
            best_params=artifact_json(classifier_selected_params),
            notes="Base model fitted on model-selection rows; sigmoid calibration fitted on the calibration block and evaluated on holdout.",
        )
    except Exception as exc:
        model_errors.append({"model": "XGBoost[GPU top-quintile classifier]", "stage": "fit_or_predict", "error": short_error(exc)})
        cleanup_runtime_memory("xgb_classifier_error")

# GPU XGBoost ranker.
xgb_ranker = None
if RUN_XGBOOST_RANKER:
    try:
        ranker_search = ProgressRandomizedSearchCV(
            make_xgb_ranker(),
            RANKER_PARAM_DISTRIBUTIONS,
            XGBOOST_RANKER_TUNING_ITERATIONS,
            cv_splits,
            label="GPU XGBoost ranker",
        )
        ranker_search.fit_ranker(X_selection, y_selection_rank, selection_df)
        ranker_trials = ranker_search.cv_results_frame()
        save_artifact_table(ranker_trials, "xgboost_ranker_tuning_trials.csv", description="GPU XGBoost ranker randomized-search trial history, including fold scores, fold errors, and trial timing.")
        if not np.isfinite(ranker_search.best_score_):
            raise RuntimeError("GPU XGBoost ranker search produced no finite chronological CV score; inspect xgboost_ranker_tuning_trials.csv before fitting a final model.")
        ranker_selected_params = dict(ranker_search.best_params_)
        ranker_selected_params["n_estimators"] = ranker_search.best_n_estimators_
        save_artifact_table(pd.DataFrame([ranker_selected_params]), "xgboost_ranker_selected_params.csv", description="Selected GPU XGBoost ranker hyperparameters from chronological randomized search, including CV-derived effective estimator count.")

        selection_sorted = selection_df.sort_values(["date", "symbol"])
        preholdout_sorted = preholdout_df.sort_values(["date", "symbol"])
        calibration_sorted = calibration_df.sort_values(["date", "symbol"])

        xgb_ranker_selection = make_xgb_ranker()
        xgb_ranker_selection.set_params(**ranker_selected_params)
        start = time.time()
        xgb_ranker_selection.fit(
            X_selection.loc[selection_sorted.index],
            y_selection_rank.loc[selection_sorted.index],
            group=group_sizes_by_month(selection_sorted),
            verbose=False,
        )
        selection_fit_seconds = time.time() - start
        start = time.time()
        cal_score = predict_scores_in_chunks(xgb_ranker_selection, X_calibration, desc="Predicting XGBoost ranker calibration")
        cal_seconds = time.time() - start

        xgb_ranker = make_xgb_ranker()
        xgb_ranker.set_params(**ranker_selected_params)
        start = time.time()
        xgb_ranker.fit(
            X_preholdout.loc[preholdout_sorted.index],
            y_preholdout_rank.loc[preholdout_sorted.index],
            group=group_sizes_by_month(preholdout_sorted),
            verbose=False,
        )
        preholdout_fit_seconds = time.time() - start
        start = time.time()
        holdout_score = predict_scores_in_chunks(xgb_ranker, X_holdout, desc="Predicting XGBoost ranker holdout")
        holdout_seconds = time.time() - start

        add_predictions(
            "XGBoost[GPU ranker]",
            "XGBRanker",
            "gpu_learning_to_rank",
            "raw_score",
            [("calibration", calibration_df, y_calibration, cal_score, cal_seconds), ("holdout", holdout_df, y_holdout, holdout_score, holdout_seconds)],
            fit_seconds=selection_fit_seconds + preholdout_fit_seconds,
            tuning_score=ranker_search.best_score_,
            best_params=artifact_json(ranker_selected_params),
            notes="Rank-aware XGBoost model using within-month relevance labels and monthly groups. Calibration and holdout scores use separate chronological fits.",
        )
    except Exception as exc:
        model_errors.append({"model": "XGBoost[GPU ranker]", "stage": "fit_or_predict", "error": short_error(exc)})
        cleanup_runtime_memory("xgb_ranker_error")

# Direct TabPFN top-quintile scorer.
def load_tabpfn_token():
    for name in ["TABPFN_TOKEN", "PRIORLABS_API_KEY", "TABPFN_API_KEY"]:
        value = os.environ.get(name)
        if value:
            os.environ["TABPFN_TOKEN"] = value.strip().strip('"').strip("'")
            os.environ.setdefault("PRIORLABS_API_KEY", os.environ["TABPFN_TOKEN"])
            return os.environ["TABPFN_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in ["TABPFN_TOKEN", "PRIORLABS_API_KEY", "TABPFN_API_KEY"]:
            value = client.get_secret(name)
            if value:
                os.environ["TABPFN_TOKEN"] = str(value).strip().strip('"').strip("'")
                os.environ.setdefault("PRIORLABS_API_KEY", os.environ["TABPFN_TOKEN"])
                return os.environ["TABPFN_TOKEN"]
    except Exception:
        return None
    return None


def sample_tabpfn_context(X, y, max_rows=TABPFN_CONTEXT_MAX_ROWS):
    frame = pd.DataFrame({"row": np.arange(len(y)), "y": y_to_numpy(y)})
    if len(frame) <= max_rows:
        return np.arange(len(y))
    positive = frame[frame["y"] == 1]
    negative = frame[frame["y"] == 0]
    positive_target = min(len(positive), max(max_rows // 3, int(max_rows * float(frame["y"].mean()))))
    negative_target = max_rows - positive_target
    sampled = pd.concat(
        [
            positive.sample(n=positive_target, random_state=SEED) if len(positive) > positive_target else positive,
            negative.sample(n=min(len(negative), negative_target), random_state=SEED) if len(negative) > negative_target else negative,
        ],
        ignore_index=True,
    )
    return sampled["row"].sort_values().to_numpy()


def make_tabpfn_classifier():
    from tabpfn import TabPFNClassifier
    from tabpfn.constants import ModelVersion
    model_version = getattr(ModelVersion, "V2_6")
    return TabPFNClassifier.create_default_for_version(model_version, n_estimators=N_TFM_ESTIMATORS, device=TABPFN_DEVICE, random_state=SEED)


if RUN_DIRECT_TABPFN:
    tabpfn_status_rows = []
    token = load_tabpfn_token()
    os.environ.setdefault("TABPFN_DISABLE_TELEMETRY", "1")
    if not token:
        tabpfn_status_rows.append({"check": "token", "status": "skipped", "evidence": "TABPFN_TOKEN was not available; TabPFN scoring skipped."})
    else:
        tabpfn_status_rows.append({"check": "token", "status": "available", "evidence": f"token_length={len(token)}"})
        try:
            context_rows = sample_tabpfn_context(X_preholdout, y_preholdout, TABPFN_CONTEXT_MAX_ROWS)
            start = time.time()
            tabpfn_model = make_tabpfn_classifier()
            tabpfn_model.fit(to_numpy_float32(X_preholdout.iloc[context_rows]), y_preholdout.iloc[context_rows].to_numpy(dtype=np.int32))
            fit_seconds = time.time() - start
            start = time.time()
            holdout_score = predict_proba_in_chunks(tabpfn_model, X_holdout, chunk_size=2048, desc="Predicting TabPFN holdout")
            holdout_seconds = time.time() - start
            add_predictions(
                "TabPFN[Direct top-quintile scorer]",
                "TabPFN 2.6",
                "tabular_foundation_model",
                "probability",
                [("holdout", holdout_df, y_holdout, holdout_score, holdout_seconds)],
                fit_seconds=fit_seconds,
                best_params=artifact_json({"n_estimators": N_TFM_ESTIMATORS, "context_rows": int(len(context_rows)), "device": safe_string(TABPFN_DEVICE)}),
                notes="Direct TabPFN classifier fitted on sampled pre-holdout context rows.",
            )
            tabpfn_status_rows.append({"check": "fit_predict", "status": "complete", "evidence": f"context_rows={len(context_rows)}; holdout_rows={len(X_holdout)}"})
            del tabpfn_model
            cleanup_runtime_memory("after_tabpfn")
        except Exception as exc:
            status = "skipped" if ALLOW_TABPFN_SKIP_IF_UNAVAILABLE else "failed"
            tabpfn_status_rows.append({"check": "fit_predict", "status": status, "evidence": short_error(exc)})
            cleanup_runtime_memory("tabpfn_error")
            if not ALLOW_TABPFN_SKIP_IF_UNAVAILABLE:
                model_errors.append({"model": "TabPFN[Direct top-quintile scorer]", "stage": "fit_or_predict", "error": short_error(exc)})
                raise
    tabpfn_status = pd.DataFrame(tabpfn_status_rows)
    save_artifact_table(tabpfn_status, "tabpfn_status_summary.csv", description="TabPFN token and execution status without exposing token values.")
else:
    tabpfn_status = pd.DataFrame(
        [
            {
                "check": "enabled",
                "status": "disabled",
                "evidence": "RUN_DIRECT_TABPFN is False; TabPFN scoring was disabled by configuration or CUDA preflight.",
            }
        ]
    )
    save_artifact_table(tabpfn_status, "tabpfn_status_summary.csv", description="TabPFN disabled status for this run.")


# Direct TabICL top-quintile scorer.
def load_huggingface_token_for_tabicl():
    for name in ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_HUB_TOKEN"]:
        value = os.environ.get(name)
        if value:
            token = str(value).strip().strip('\"').strip("'")
            os.environ["HF_TOKEN"] = token
            os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", token)
            return token
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_HUB_TOKEN"]:
            value = client.get_secret(name)
            if value:
                token = str(value).strip().strip('\"').strip("'")
                os.environ["HF_TOKEN"] = token
                os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", token)
                return token
    except Exception:
        return None
    return None


def tabicl_batch_size(n_rows):
    if CUDA_DEVICE_COUNT == 0:
        return 256
    if FAST_MODE:
        return 512
    return 1024 if n_rows <= 5_000 else 512


def make_tabicl_classifier(n_rows):
    from tabicl import TabICLClassifier
    return TabICLClassifier(
        n_estimators=N_TABICL_ESTIMATORS,
        device=TABICL_DEVICE,
        batch_size=tabicl_batch_size(n_rows),
        random_state=SEED,
        checkpoint_version=TABICL_CHECKPOINT_VERSION,
    )


if RUN_DIRECT_TABICL:
    tabicl_status_rows = []
    hf_token = load_huggingface_token_for_tabicl()
    if hf_token:
        tabicl_status_rows.append({"check": "huggingface_token", "status": "available", "evidence": f"token_length={len(hf_token)}"})
    else:
        tabicl_status_rows.append({"check": "huggingface_token", "status": "not_found", "evidence": "No HF token was found; TabICL will still run if the checkpoint is public or already cached."})
    try:
        context_rows = sample_tabpfn_context(X_preholdout, y_preholdout, TABICL_CONTEXT_MAX_ROWS)
        start = time.time()
        tabicl_model = make_tabicl_classifier(len(context_rows))
        tabicl_model.fit(to_numpy_float32(X_preholdout.iloc[context_rows]), y_preholdout.iloc[context_rows].to_numpy(dtype=np.int32))
        fit_seconds = time.time() - start
        start = time.time()
        holdout_score = predict_proba_in_chunks(tabicl_model, X_holdout, chunk_size=2048, desc="Predicting TabICL holdout")
        holdout_seconds = time.time() - start
        add_predictions(
            "TabICL[Direct top-quintile scorer]",
            "TabICL",
            "tabular_foundation_model",
            "probability",
            [("holdout", holdout_df, y_holdout, holdout_score, holdout_seconds)],
            fit_seconds=fit_seconds,
            best_params=artifact_json(
                {
                    "n_estimators": N_TABICL_ESTIMATORS,
                    "context_rows": int(len(context_rows)),
                    "device": safe_string(TABICL_DEVICE),
                    "batch_size": tabicl_batch_size(len(context_rows)),
                    "checkpoint_version": TABICL_CHECKPOINT_VERSION,
                }
            ),
            notes="Direct TabICL classifier fitted on sampled pre-holdout context rows with memory-aware context limits.",
        )
        tabicl_status_rows.append({"check": "holdout_fit_predict", "status": "complete", "evidence": f"context_rows={len(context_rows)}; holdout_rows={len(X_holdout)}"})
        del tabicl_model
        cleanup_runtime_memory("after_tabicl_holdout")

        selection_context_rows = sample_tabpfn_context(X_selection, y_selection, min(TABICL_CONTEXT_MAX_ROWS, len(X_selection)))
        start = time.time()
        tabicl_calibration_base = make_tabicl_classifier(len(selection_context_rows))
        tabicl_calibration_base.fit(to_numpy_float32(X_selection.iloc[selection_context_rows]), y_selection.iloc[selection_context_rows].to_numpy(dtype=np.int32))
        calibration_base_fit_seconds = time.time() - start
        start = time.time()
        calibration_score = predict_proba_in_chunks(tabicl_calibration_base, X_calibration, chunk_size=2048, desc="Predicting TabICL calibration")
        calibration_seconds = time.time() - start
        start = time.time()
        calibration_base_holdout_score = predict_proba_in_chunks(tabicl_calibration_base, X_holdout, chunk_size=2048, desc="Predicting TabICL calibration-base holdout")
        calibration_base_holdout_seconds = time.time() - start
        add_predictions(
            "TabICL[Selection-fit calibration base]",
            "TabICL",
            "tabular_foundation_model",
            "probability",
            [
                ("calibration", calibration_df, y_calibration, calibration_score, calibration_seconds),
                ("holdout", holdout_df, y_holdout, calibration_base_holdout_score, calibration_base_holdout_seconds),
            ],
            fit_seconds=calibration_base_fit_seconds,
            best_params=artifact_json(
                {
                    "n_estimators": N_TABICL_ESTIMATORS,
                    "context_rows": int(len(selection_context_rows)),
                    "device": safe_string(TABICL_DEVICE),
                    "batch_size": tabicl_batch_size(len(selection_context_rows)),
                    "checkpoint_version": TABICL_CHECKPOINT_VERSION,
                    "training_window": "selection_only",
                }
            ),
            notes="Direct TabICL classifier fitted on model-selection rows so calibration-window diagnostics remain out of sample.",
        )
        tabicl_status_rows.append({"check": "calibration_base_fit_predict", "status": "complete", "evidence": f"context_rows={len(selection_context_rows)}; calibration_rows={len(X_calibration)}; holdout_rows={len(X_holdout)}"})
        del tabicl_calibration_base
        cleanup_runtime_memory("after_tabicl_calibration_base")
    except Exception as exc:
        status = "skipped" if ALLOW_TABICL_SKIP_IF_UNAVAILABLE else "failed"
        tabicl_status_rows.append({"check": "fit_predict", "status": status, "evidence": short_error(exc)})
        cleanup_runtime_memory("tabicl_error")
        if not ALLOW_TABICL_SKIP_IF_UNAVAILABLE:
            model_errors.append({"model": "TabICL[Direct top-quintile scorer]", "stage": "fit_or_predict", "error": short_error(exc)})
            raise
    tabicl_status = pd.DataFrame(tabicl_status_rows)
    save_artifact_table(tabicl_status, "tabicl_status_summary.csv", description="TabICL authentication, checkpoint, and execution status without exposing token values.")
else:
    tabicl_status = pd.DataFrame(
        [
            {
                "check": "enabled",
                "status": "disabled",
                "evidence": "RUN_DIRECT_TABICL is False; TabICL scoring was disabled by configuration or CUDA preflight.",
            }
        ]
    )
    save_artifact_table(tabicl_status, "tabicl_status_summary.csv", description="TabICL disabled status for this run.")


# Fundamental augmented diagnostic, separate from headline model comparison.
if RUN_FUNDAMENTAL_DIAGNOSTIC and RUN_FUNDAMENTAL_XGBOOST_DIAGNOSTIC and len(fundamental_feature_columns) > 0 and RUN_XGBOOST_CLASSIFIER:
    try:
        diagnostic_columns = sorted(set(ACTIVE_FEATURE_COLUMNS + [column for column in fundamental_feature_columns if column in model_frame_with_sector.columns]))
        diag_imputer = SimpleImputer(strategy="median", add_indicator=True)
        X_diag_preholdout = pd.DataFrame(diag_imputer.fit_transform(preholdout_df.reindex(columns=diagnostic_columns)), index=preholdout_df.index)
        X_diag_holdout = pd.DataFrame(diag_imputer.transform(holdout_df.reindex(columns=diagnostic_columns)), index=holdout_df.index)
        diagnostic_model = make_xgb_classifier(n_estimators=FUNDAMENTAL_DIAGNOSTIC_N_ESTIMATORS, y_for_weight=y_preholdout)
        diagnostic_model.set_params(max_depth=3, learning_rate=0.03, subsample=0.85, colsample_bytree=0.75, min_child_weight=10.0, reg_lambda=20.0, reg_alpha=0.1)
        start = time.time()
        diagnostic_model.fit(X_diag_preholdout, y_preholdout, verbose=False)
        fit_seconds = time.time() - start
        start = time.time()
        holdout_score = predict_proba_in_chunks(diagnostic_model, X_diag_holdout, desc="Predicting fundamental diagnostic XGBoost")
        holdout_seconds = time.time() - start
        add_predictions(
            "XGBoost[Fundamental diagnostic]",
            "XGBClassifier fixed",
            "diagnostic_only",
            "probability",
            [("holdout", holdout_df, y_holdout, holdout_score, holdout_seconds)],
            fit_seconds=fit_seconds,
            best_params=artifact_json(diagnostic_model.get_params()),
            notes="Diagnostic only; public fundamentals are not point-in-time headline evidence.",
        )
    except Exception as exc:
        model_errors.append({"model": "XGBoost[Fundamental diagnostic]", "stage": "fit_or_predict", "error": short_error(exc)})
        cleanup_runtime_memory("fundamental_diagnostic_error")

model_summary = pd.DataFrame(model_rows)
if len(model_summary) > 0:
    save_artifact_table(model_summary, "model_metric_summary.csv", description="Row-level model metrics by evaluation window.")
    display(model_summary.sort_values(["Evaluation Window", "Average Precision"], ascending=[True, False]).head(20))

if model_errors:
    model_error_summary = pd.DataFrame(model_errors)
else:
    model_error_summary = pd.DataFrame(columns=["model", "stage", "error"])
save_artifact_table(model_error_summary, "model_error_summary.csv", description="Model errors or skipped paths recorded during execution.")

if prediction_frames:
    all_predictions = pd.concat(prediction_frames, ignore_index=True)
else:
    raise RuntimeError("No prediction frames were produced.")
if SAVE_FULL_PREDICTION_SCORES:
    save_artifact_table(all_predictions, "all_model_predictions.csv", description="Full prediction table for all evaluated model scores.")
print(f"Saved prediction rows: {len(all_predictions):,}")

## 6. Ranking, Calibration, Portfolio, Cost, and Robustness Diagnostics

Row-level model metrics are not sufficient for a cross-sectional ranking problem. This section evaluates monthly rank IC, precision at top-k, top-minus-bottom returns, long-only and long-short score translation, turnover, transaction-cost sensitivity, calibration, sector exposure, and liquidity proxies.

In [ ]:
# Monthly ranking diagnostics.
ranking_rows = []
for (model_name, evaluation_window), group in tqdm(all_predictions.groupby(["Model", "Evaluation Window"]), desc="Computing ranking diagnostics", unit="model-window"):
    diagnostics = ranking_diagnostics(group)
    if len(diagnostics) == 0:
        continue
    diagnostics["Model"] = model_name
    diagnostics["Evaluation Window"] = evaluation_window
    ranking_rows.append(diagnostics)
ranking_summary = pd.concat(ranking_rows, ignore_index=True) if ranking_rows else pd.DataFrame()
aggregate_ranking_summary = pd.DataFrame()
yearly_ranking_summary = pd.DataFrame()
regime_ranking_summary = pd.DataFrame()
if len(ranking_summary) > 0:
    save_artifact_table(ranking_summary, "monthly_ranking_diagnostics.csv", description="Monthly cross-sectional rank, precision, and score-to-return diagnostics.")
    aggregate_ranking_summary = (
        ranking_summary.groupby(["Model", "Evaluation Window"])
        .agg(
            months=("date", "count"),
            mean_rank_ic=("rank_ic", "mean"),
            median_rank_ic=("rank_ic", "median"),
            mean_average_precision=("average_precision", "mean"),
            mean_precision_at_top_k=("precision_at_top_k", "mean"),
            mean_top_return=("top_mean_return", "mean"),
            mean_bottom_return=("bottom_mean_return", "mean"),
            mean_top_minus_bottom_return=("top_minus_bottom_return", "mean"),
            mean_top_excess_return=("top_excess_return", "mean"),
        )
        .reset_index()
    )
    save_artifact_table(aggregate_ranking_summary, "aggregate_ranking_summary.csv", description="Aggregated monthly ranking diagnostics.")
    display(aggregate_ranking_summary.sort_values(["Evaluation Window", "mean_rank_ic"], ascending=[True, False]).head(20))

    ranking_summary["year"] = pd.to_datetime(ranking_summary["date"]).dt.year
    yearly_ranking_summary = (
        ranking_summary.groupby(["Model", "Evaluation Window", "year"])
        .agg(
            months=("date", "count"),
            mean_rank_ic=("rank_ic", "mean"),
            mean_average_precision=("average_precision", "mean"),
            mean_precision_at_top_k=("precision_at_top_k", "mean"),
            mean_top_minus_bottom_return=("top_minus_bottom_return", "mean"),
            mean_top_excess_return=("top_excess_return", "mean"),
        )
        .reset_index()
    )
    save_artifact_table(yearly_ranking_summary, "yearly_ranking_summary.csv", description="Year-by-year ranking diagnostics for stability review.")

    regime_reference = holdout_df[["date"]].drop_duplicates().copy()
    benchmark_monthly = model_frame_with_sector[model_frame_with_sector["symbol"] == model_symbols[0]][["date"]].drop_duplicates().copy()
    market_regime_columns = [column for column in model_frame_with_sector.columns if column.startswith("market_spy_return_63d") or column.startswith("macro_vix_close")]
    if market_regime_columns:
        regime_source = model_frame_with_sector[["date"] + market_regime_columns].drop_duplicates("date").copy()
        regime_reference = regime_reference.merge(regime_source, on="date", how="left")
        if "market_spy_return_63d" in regime_reference.columns:
            regime_reference["spy_63d_regime"] = np.where(regime_reference["market_spy_return_63d"] >= 0.0, "spy_63d_nonnegative", "spy_63d_negative")
        else:
            regime_reference["spy_63d_regime"] = "unavailable"
        if "macro_vix_close" in regime_reference.columns and regime_reference["macro_vix_close"].notna().any():
            vix_threshold = regime_reference["macro_vix_close"].median(skipna=True)
            regime_reference["vix_regime"] = np.where(regime_reference["macro_vix_close"] >= vix_threshold, "higher_vix", "lower_vix")
        else:
            regime_reference["vix_regime"] = "unavailable"
        regime_long = []
        for regime_column in ["spy_63d_regime", "vix_regime"]:
            merged = ranking_summary.merge(regime_reference[["date", regime_column]], on="date", how="left")
            summary = (
                merged.groupby(["Model", "Evaluation Window", regime_column])
                .agg(
                    months=("date", "count"),
                    mean_rank_ic=("rank_ic", "mean"),
                    mean_average_precision=("average_precision", "mean"),
                    mean_precision_at_top_k=("precision_at_top_k", "mean"),
                    mean_top_minus_bottom_return=("top_minus_bottom_return", "mean"),
                    mean_top_excess_return=("top_excess_return", "mean"),
                )
                .reset_index()
                .rename(columns={regime_column: "regime_value"})
            )
            summary["regime_type"] = regime_column
            regime_long.append(summary)
        regime_ranking_summary = pd.concat(regime_long, ignore_index=True) if regime_long else pd.DataFrame()
        if len(regime_ranking_summary) > 0:
            save_artifact_table(regime_ranking_summary, "regime_ranking_summary.csv", description="Ranking diagnostics by simple SPY and VIX market-regime buckets.")

# Calibration diagnostics for probability models.
calibration_rows = []
for (model_name, evaluation_window), group in tqdm(all_predictions[all_predictions["Score Type"] == "probability"].groupby(["Model", "Evaluation Window"]), desc="Computing calibration tables", unit="model-window"):
    table = calibration_bin_table(group["target_top_quintile"].astype(int), group["score"], n_bins=10)
    table["Model"] = model_name
    table["Evaluation Window"] = evaluation_window
    calibration_rows.append(table)
calibration_summary = pd.concat(calibration_rows, ignore_index=True) if calibration_rows else pd.DataFrame()
if len(calibration_summary) > 0:
    save_artifact_table(calibration_summary, "calibration_bin_summary.csv", description="Probability calibration bins for probability-scoring models.")

# Portfolio translation and cost sensitivity.
portfolio_rows = []
portfolio_sensitivity_rows = []
for model_name, group in tqdm(all_predictions[all_predictions["Evaluation Window"] == "holdout"].groupby("Model"), desc="Computing portfolio diagnostics", unit="model"):
    portfolio = score_to_portfolio_returns(group, model_name, transaction_cost_bps=TRANSACTION_COST_BPS)
    if len(portfolio) == 0:
        continue
    portfolio_rows.append(portfolio)
    for bps in TRANSACTION_COST_SENSITIVITY_BPS:
        sensitivity = score_to_portfolio_returns(group, model_name, transaction_cost_bps=bps)
        long_summary = annualized_summary(sensitivity["long_top_return_net"], f"{model_name} long_top", bps)
        long_short_summary = annualized_summary(sensitivity["long_short_return_net"], f"{model_name} long_short", bps)
        long_summary.update({"Model": model_name, "portfolio": "long_top"})
        long_short_summary.update({"Model": model_name, "portfolio": "long_short"})
        portfolio_sensitivity_rows.extend([long_summary, long_short_summary])
    sector_neutral_default = score_to_sector_neutral_portfolio_returns(group, model_name, transaction_cost_bps=TRANSACTION_COST_BPS)
    if len(sector_neutral_default) > 0:
        portfolio_rows.append(sector_neutral_default)
    for bps in TRANSACTION_COST_SENSITIVITY_BPS:
        sector_neutral_sensitivity = score_to_sector_neutral_portfolio_returns(group, model_name, transaction_cost_bps=bps)
        if len(sector_neutral_sensitivity) == 0:
            continue
        sector_long_summary = annualized_summary(sector_neutral_sensitivity["long_top_return_net"], f"{model_name} sector_neutral_long_top", bps)
        sector_long_short_summary = annualized_summary(sector_neutral_sensitivity["long_short_return_net"], f"{model_name} sector_neutral_long_short", bps)
        sector_long_summary.update({"Model": f"{model_name} sector-neutral", "portfolio": "sector_neutral_long_top"})
        sector_long_short_summary.update({"Model": f"{model_name} sector-neutral", "portfolio": "sector_neutral_long_short"})
        portfolio_sensitivity_rows.extend([sector_long_summary, sector_long_short_summary])
equal_weight_default = equal_weight_universe_returns(holdout_df, transaction_cost_bps=TRANSACTION_COST_BPS)
if len(equal_weight_default) > 0:
    portfolio_rows.append(equal_weight_default)
for bps in TRANSACTION_COST_SENSITIVITY_BPS:
    equal_weight_sensitivity = equal_weight_universe_returns(holdout_df, transaction_cost_bps=bps)
    if len(equal_weight_sensitivity) > 0:
        equal_weight_summary = annualized_summary(equal_weight_sensitivity["long_top_return_net"], "Benchmark[Equal-weight universe] long_only", bps)
        equal_weight_summary.update({"Model": "Benchmark[Equal-weight universe]", "portfolio": "long_only"})
        portfolio_sensitivity_rows.append(equal_weight_summary)

portfolio_returns = pd.concat(portfolio_rows, ignore_index=True) if portfolio_rows else pd.DataFrame()
portfolio_sensitivity_summary = pd.DataFrame(portfolio_sensitivity_rows)
if len(portfolio_returns) > 0:
    save_artifact_table(portfolio_returns, "portfolio_monthly_returns.csv", description="Monthly score-to-portfolio returns, turnover, and liquidity proxies.")
if len(portfolio_sensitivity_summary) > 0:
    save_artifact_table(portfolio_sensitivity_summary, "portfolio_transaction_cost_sensitivity.csv", description="Portfolio annualized metrics across transaction-cost assumptions.")
    display(portfolio_sensitivity_summary[(portfolio_sensitivity_summary["transaction_cost_bps"] == TRANSACTION_COST_BPS)].sort_values("sharpe", ascending=False).head(20))

active_return_rows = []
if len(portfolio_returns) > 0:
    benchmark_returns = portfolio_returns.loc[portfolio_returns["Model"] == "Benchmark[Equal-weight universe]", ["date", "long_top_return_net"]].rename(columns={"long_top_return_net": "benchmark_equal_weight_return_net"})
    for model_name, group in portfolio_returns[portfolio_returns["Model"] != "Benchmark[Equal-weight universe]"].groupby("Model"):
        merged = group.merge(benchmark_returns, on="date", how="inner")
        if len(merged) == 0:
            continue
        active_long = merged["long_top_return_net"] - merged["benchmark_equal_weight_return_net"]
        active_summary = annualized_summary(active_long, f"{model_name} active_vs_equal_weight", TRANSACTION_COST_BPS)
        active_summary.update({"Model": model_name, "benchmark": "Benchmark[Equal-weight universe]", "portfolio": "long_top_active"})
        active_return_rows.append(active_summary)
active_return_summary = pd.DataFrame(active_return_rows)
if len(active_return_summary) > 0:
    save_artifact_table(active_return_summary, "portfolio_active_return_vs_equal_weight.csv", description="Holdout active return diagnostics versus equal-weight universe benchmark.")

liquidity_participation_summary = pd.DataFrame()
if len(portfolio_returns) > 0:
    liquidity_columns = [column for column in portfolio_returns.columns if "participation" in column or "trade_notional" in column or column in ["date", "Model", "avg_top_dollar_volume_63d", "avg_top_high_low_range_21d"]]
    liquidity_participation_summary = portfolio_returns[liquidity_columns].copy()
    save_artifact_table(liquidity_participation_summary, "portfolio_liquidity_participation_summary.csv", description="Simple ADV participation and high-low range proxies for selected score portfolios.")

# Bootstrap uncertainty for holdout monthly rank IC and top-minus-bottom return.
bootstrap_rows = []
if len(ranking_summary) > 0:
    holdout_ranking = ranking_summary[ranking_summary["Evaluation Window"] == "holdout"].copy()
    rng = np.random.default_rng(SEED)
    for model_name, group in tqdm(holdout_ranking.groupby("Model"), desc="Bootstrap ranking uncertainty", unit="model"):
        values = group[["rank_ic", "top_minus_bottom_return", "precision_at_top_k"]].dropna()
        if len(values) < 3:
            continue
        for metric in ["rank_ic", "top_minus_bottom_return", "precision_at_top_k"]:
            samples = []
            metric_values = values[metric].to_numpy(dtype=float)
            for _ in range(BOOTSTRAP_ITERATIONS):
                draw = rng.choice(metric_values, size=len(metric_values), replace=True)
                samples.append(float(np.nanmean(draw)))
            bootstrap_rows.append(
                {
                    "Model": model_name,
                    "metric": metric,
                    "months": len(metric_values),
                    "mean": float(np.nanmean(metric_values)),
                    "bootstrap_mean": float(np.nanmean(samples)),
                    "ci_lower_2_5pct": float(np.nanpercentile(samples, 2.5)),
                    "ci_upper_97_5pct": float(np.nanpercentile(samples, 97.5)),
                }
            )
bootstrap_summary = pd.DataFrame(bootstrap_rows)
if len(bootstrap_summary) > 0:
    save_artifact_table(bootstrap_summary, "bootstrap_uncertainty_summary.csv", description="Bootstrap uncertainty for holdout monthly ranking diagnostics.")

# Sector and liquidity exposure diagnostics for selected top baskets.
exposure_rows = []
for model_name, group in tqdm(all_predictions[all_predictions["Evaluation Window"] == "holdout"].groupby("Model"), desc="Computing exposure diagnostics", unit="model"):
    for date, month_group in group.dropna(subset=["score"]).groupby("date", sort=True):
        month_group = month_group.sort_values(["score", "symbol"], ascending=[False, True])
        top_k = max(1, int(math.ceil(len(month_group) * TOP_QUANTILE))) if TOP_K_PORTFOLIO is None else min(TOP_K_PORTFOLIO, len(month_group))
        top = month_group.head(top_k)
        sector_counts = top["sector"].value_counts(normalize=True).to_dict() if "sector" in top else {}
        row = {"Model": model_name, "date": date, "top_k": top_k, "sector_hhi": float(sum(value**2 for value in sector_counts.values())) if sector_counts else np.nan}
        for sector, weight in sector_counts.items():
            clean_sector = str(sector).lower().replace(" ", "_").replace("/", "_")[:40]
            row[f"sector_weight_{clean_sector}"] = weight
        exposure_rows.append(row)
sector_exposure_summary = pd.DataFrame(exposure_rows)
if len(sector_exposure_summary) > 0:
    save_artifact_table(sector_exposure_summary, "sector_exposure_diagnostics.csv", description="Sector concentration diagnostics for selected top baskets.")

# Figures.
if len(aggregate_ranking_summary) > 0:
    holdout_aggregate = aggregate_ranking_summary[aggregate_ranking_summary["Evaluation Window"] == "holdout"].sort_values("mean_rank_ic", ascending=False)
    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(holdout_aggregate))))
    ax.barh(holdout_aggregate["Model"], holdout_aggregate["mean_rank_ic"])
    ax.axvline(0.0, color="black", linewidth=0.8)
    ax.set_title("Holdout mean monthly rank IC")
    ax.set_xlabel("Spearman rank correlation")
    ax.invert_yaxis()
    save_figure_artifact(fig, "holdout_rank_ic_by_model.png", description="Holdout mean monthly rank IC by model.")
    plt.show()

if len(portfolio_returns) > 0:
    fig, ax = plt.subplots(figsize=(11, 6))
    for model_name, group in portfolio_returns.groupby("Model"):
        ordered = group.sort_values("date")
        equity = (1.0 + ordered["long_top_return_net"].fillna(0.0)).cumprod()
        ax.plot(ordered["date"], equity, label=model_name, linewidth=1.4)
    ax.set_title("Holdout long-top basket equity curves, net of configured cost")
    ax.set_ylabel("Growth of 1.0")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.legend(fontsize=8, loc="best")
    ax.grid(True, alpha=0.25)
    save_figure_artifact(fig, "portfolio_equity_curves_holdout.png", description="Holdout score-to-portfolio equity curves.")
    plt.show()

if len(calibration_summary) > 0:
    holdout_calibration = calibration_summary[calibration_summary["Evaluation Window"] == "holdout"]
    fig, ax = plt.subplots(figsize=(7, 6))
    for model_name, group in holdout_calibration.groupby("Model"):
        ax.plot(group["mean_predicted_probability"], group["observed_rate"], marker="o", label=model_name)
    ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1.0)
    ax.set_title("Holdout calibration by quantile bin")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed positive rate")
    ax.legend(fontsize=8, loc="best")
    ax.grid(True, alpha=0.25)
    save_figure_artifact(fig, "calibration_curves_holdout.png", description="Holdout calibration curves for probability models.")
    plt.show()

## 7. Run Closeout and Output Archive

The final section writes compact text summaries, records the complete artifact manifest, and creates a ZIP archive for download from Kaggle.

In [ ]:
headline_lines = [
    "# Cross-Sectional Equity Return Ranking Run Summary",
    "",
    "Research only. Not investment advice.",
    "",
    f"Run mode: {'FAST_MODE' if FAST_MODE else 'production research mode'}",
    f"Rows in final model frame: {len(model_frame):,}",
    f"Months in final model frame: {model_frame['date'].nunique()}",
    f"Symbols in final model frame: {model_frame['symbol'].nunique()}",
    f"Headline feature count before imputation: {len(ACTIVE_FEATURE_COLUMNS)}",
    f"Selection/calibration feature count after imputation: {X_selection.shape[1]}",
    f"Preholdout/holdout feature count after imputation: {X_preholdout.shape[1]}",
    "",
    "## Interpretation Guardrails",
    "",
    "- The universe is a static public large-cap list and is not point-in-time index membership.",
    "- Public fundamentals are diagnostic only and are excluded from headline evidence.",
    "- Scores are evaluated as cross-sectional ranking diagnostics, not as live trading recommendations.",
    "- Portfolio translations are simplified research diagnostics with transaction-cost sensitivity.",
]
if len(model_summary) > 0:
    holdout_metrics = model_summary[model_summary["Evaluation Window"] == "holdout"].sort_values("Average Precision", ascending=False)
    headline_lines.extend(["", "## Holdout Row-Level Metrics", ""])
    headline_lines.append(holdout_metrics[["Model", "Average Precision", "ROC AUC", "Brier Score", "Workflow Seconds"]].to_markdown(index=False))
if len(aggregate_ranking_summary) > 0:
    holdout_rank = aggregate_ranking_summary[aggregate_ranking_summary["Evaluation Window"] == "holdout"].sort_values("mean_rank_ic", ascending=False)
    headline_lines.extend(["", "## Holdout Ranking Diagnostics", ""])
    headline_lines.append(holdout_rank[["Model", "mean_rank_ic", "mean_precision_at_top_k", "mean_top_minus_bottom_return", "mean_top_excess_return"]].to_markdown(index=False))
if len(portfolio_sensitivity_summary) > 0:
    default_cost = portfolio_sensitivity_summary[portfolio_sensitivity_summary["transaction_cost_bps"] == TRANSACTION_COST_BPS].sort_values("sharpe", ascending=False)
    headline_lines.extend(["", "## Holdout Portfolio Diagnostics at Configured Cost", ""])
    headline_lines.append(default_cost[["Model", "portfolio", "total_return", "cagr", "sharpe", "max_drawdown"]].to_markdown(index=False))

run_summary_text = "\n".join(headline_lines)
save_text_artifact("run_summary.md", run_summary_text, description="Compact Markdown run summary for post-run review.")
save_text_artifact("publication_notes.txt", run_summary_text, description="Plain-text publication notes; review before using in a blog post.")

run_closeout = pd.DataFrame(
    [
        {"item": "workflow_complete", "status": True, "detail": "Notebook reached archive section."},
        {"item": "prediction_rows", "status": len(all_predictions) > 0, "detail": len(all_predictions)},
        {"item": "model_error_rows", "status": len(model_error_summary) == 0, "detail": len(model_error_summary)},
        {"item": "fundamentals_headline_excluded", "status": True, "detail": "Fundamentals used only in diagnostic path when available."},
        {"item": "future_eligibility_features_excluded", "status": not any(column.startswith("cs_rank_monthly_eligible_assets") or column == "monthly_eligible_assets" for column in ACTIVE_FEATURE_COLUMNS), "detail": "Future availability counts are not model features."},
        {"item": "xgboost_best_iteration_recorded", "status": (not RUN_XGBOOST_CLASSIFIER) or ("fold_best_iterations" in (classifier_trials.columns if "classifier_trials" in globals() else [])), "detail": "Classifier search records fold best iterations when XGBoost ran; skipped only when classifier path is disabled."},
        {"item": "tabpfn_status_artifact_saved", "status": (ARTIFACT_DIR / "tabpfn_status_summary.csv").exists(), "detail": "Saved for enabled, skipped, failed, or disabled TabPFN paths."},
        {"item": "tabicl_path_configured", "status": "RUN_DIRECT_TABICL" in globals(), "detail": f"RUN_DIRECT_TABICL={RUN_DIRECT_TABICL}; TABICL_CONTEXT_MAX_ROWS={TABICL_CONTEXT_MAX_ROWS}; checkpoint={TABICL_CHECKPOINT_VERSION}"},
        {"item": "tabicl_status_artifact_saved", "status": (ARTIFACT_DIR / "tabicl_status_summary.csv").exists(), "detail": "Saved for enabled, skipped, failed, or disabled TabICL paths."},
        {"item": "artifact_directory", "status": ARTIFACT_DIR.exists(), "detail": str(ARTIFACT_DIR)},
    ]
)
save_artifact_table(run_closeout, "run_closeout.csv", description="Final closeout checklist.")

registered_artifact_manifest = pd.DataFrame(artifact_manifest_rows).drop_duplicates(subset=["path", "kind"], keep="last")
if len(registered_artifact_manifest) > 0:
    save_artifact_table(registered_artifact_manifest, "registered_artifact_manifest.csv", description="Registered artifacts written by notebook helper functions.")

artifact_file_rows = []
for artifact_path in tqdm(sorted(ARTIFACT_DIR.glob("**/*")), desc="Building artifact manifest", unit="path"):
    if artifact_path.is_file():
        artifact_file_rows.append(
            {
                "path": str(artifact_path),
                "filename": artifact_path.name,
                "suffix": artifact_path.suffix,
                "size_bytes": artifact_path.stat().st_size,
            }
        )
artifact_file_manifest = pd.DataFrame(artifact_file_rows)
if len(artifact_file_manifest) > 0:
    save_artifact_table(artifact_file_manifest, "artifact_file_manifest.csv", description="Files present in the artifact directory at archive time.")
    display(artifact_file_manifest)

archive_base_candidates = [
    Path("/kaggle/working/cross_sectional_equity_return_ranking_20260521_outputs"),
    ARTIFACT_DIR.parent / "cross_sectional_equity_return_ranking_20260521_outputs",
]
for archive_base in archive_base_candidates:
    try:
        archive_base.parent.mkdir(parents=True, exist_ok=True)
        archive_path = shutil.make_archive(str(archive_base), "zip", ARTIFACT_DIR)
        print(f"Created {archive_path}")
        break
    except Exception as exc:
        print(f"Could not create archive at {archive_base}.zip: {short_error(exc)}")

print("Notebook workflow complete. Review CSV, TXT, MD, and PNG artifacts for the run record.")